# GwenLand glcuda - T4 Ceiling Wave 10 (Factorial)

Decision-grade four-arm Kaggle sprint for the 15,000+ prefill tok/s target: retained Wave 4, Wave 8 rowCTA, Wave 9 L2 raster, and their clean combined patch. Every arm runs once in every session position.


## 1 - Configuration, clean patch stack, and T4 gate

All four worktrees start at the pinned revision. Rejected Waves 5/6/7 are structurally excluded. The combined arm is a single patch against Wave 4, avoiding patch-order ambiguity.


In [ ]:
import base64
import datetime as dt
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import re
import shutil
import statistics
import subprocess
import sys

REPO_URL = "https://github.com/gwenland-org/gwenland-ai.git"
BASE_REV = "3bce8dd7b8aaa2765855ab927c611b54981f9241"
WAVE3_PATCH_SHA256 = "5f09f6147636c36db4e23b9d5f16a3384ca4c0b69679d5443d7c9a396387a508"
WAVE3_PATCH_GZIP_B64 = """H4sIAAAAAAACCu19eXPbSJLv//oU1dqwmzRJiDh4ia3Zlm11j8NHeyTt80zoaSmQBEm0cNAAKInb44n3Id4nfJ/k5VG4CFAG17MRzVE7ukUKqEpUZWX+8qgScmrPZqLVmtuRMI/mzmQ1NY/CYHJ0awWe5YTy0ih0ex1lGT2IcYVGB7Y3tR5Ee2zqE9VUlP50PB1Mp0Jtt7uGcdBqtSo966DRaFR73o8/ipbWbfZEg37++OOBODoS1p0VrMW9GSzhRygCqxVY5tT25iJaWOLsw+Wb8zNhTiL7zoxs3xOhY47FLPBd8U4T8LsdhcK/90SLqM38gPrNzcgSP5+9f98Umt7tE/1QPAhNG4i3L8WJ+EfHaIv3L4U/QzpRYM5m9kQsrUA45sqbLBJqZtzHNaPAflDEqeOImCD0NsXY8Se3IlyYgUXPDk0Xvvi3lhc2RejjtYMWUAPirdsWt85NyJ5YotY1RODf4xh1TeAINXhoXZjelKYIt2e6dtDYgY7RJzp6GR2cXbbvxHSsUOAMPP9eXFye/nz2Wrz5IC7+fHoO396fvf/l/G/AbngC8kg+3JsSoYnvhSvXmorxOrOcx+I04es08IFb/Yem+OmnD7QuoZj74vX56fvW2F95UwUJkYToJCF6RkLw31/6o7a48E/F51CEEUiIK95cAGftENZr7a8iUfuE82655q+waCcn4uV/XsLAHL5QVySl16LWf+jDsuhaXcT/TmA9eb2Aa7C0lj1fRMRF7vYRGB4tUCxB8My5a3nAyNo88FfLN6+ht2N61pHRFJE9l789M+rHtOZCmG24FrqWOzKvalFbNITsWH8BC90QC9OZvVC78A26vzCEojSMayFqvmeJFYySlvxRMrDOFcjw/MdI5tOVlyFwfXU7fnQookhm2m6KqQqk/taKbMdCXgkYVSMeUxNZHwrtBRBq8kdDBTLRvU+sT1a7ozZVVTTwQ4vX27HmplNXxEVkzhEHYI3nKzOYgoBFPjwKxKVf82DB6rRGpGOeD7IAbXGRQjG21r6UTVTIpTklQAE5BYGcRCxsuDzn1ueVHVi0oscCxegZCABID8zP9uAX4Av/hs8TP5yg/tSQZrDyPFCEyWLl3YZEy4zEx/Ozn968ezd6eXr56s8CG9eH4mH0OTx6GEkVI910V2EEgxSn79798ur0EhRstYS50VJXHdJQWOZkIdatV5enYuKYLigYKLjlzJBLZkTEXB8eJBGhKe4XuFY7jAcZmON3zXoADnInBAtkN/AGlV7cB3YUWR6oGnLjHaHoMQiZPRUTy3ZqOJUjUD/xAphpT49ADOqIyJ2uYOUKuecFAun0WGhtArACwjWoy8sifCk05dyDa1uf3ORB0bIe4UIVxoLEzlIWg40BvAc16BotAguyRAo9R1mfqCimKBh+YIPUmo60I0wnnpPe7mk7zgnXoPXP+3cglDs7tMcgCAqIGGD13BnNLdcdua45+tyvHZDCK0szAIxVViA8y9H957BJ6trTEZx7RlM1SFupaWDNhTIGsXwW/GD0/zTMXp7h5dkPupa/PAayz4LpD4YB11t0neznVCimY889ASikjPsAeAB3KAjXQ2x0hCvFUJ0xlm/B7kRWeNCQTT6Zd5bQj4Vnok2C59lhZAWhmJgewISAB5i8VN7KHVv40AVaR9MBo0a8V5hUdmKj+XoUgQbyN28cfwNFHeZa88Tw1oPfjL+G6de1P0wG+uHsr5ew6LFNOZIqGfswoT+LwJBaraW9tBzbg4HethzfXxbHN0Yaba8pv6leblC0CPfhhBrA58btJbLg2dID3Y6vb18MlOCti0EKi2gLfZd2hF5UCTkjpvYQXoHYMzVJDodaUIIDwXScKYslSSXwU22KKxLO6yFJZ7/TBOM10JtaN5HOtA+to0ZdbO96WHpbp9sICtigIFGuf8cu3gJw9XvwRKzABk0HREADSJ4prg/gNEAwA4MCgD42QzA/CTUkgCgjncQW+eHIMR/6oaCiNBKio9EihHLNWxAM8q0SMrYHjtBqQnwaWw4OAEhaD8A+Z52HIny8Y7VogKkRTMbTRuwywQo4kb0EYACvFs0cDgPIBBYIpCX69diZhWeARpG1zdiGhFy6fK1NwwvK6JowbhGu0C20Qeiljw1CO0f7btqOALSVgwN+y4VJ9G8CTYCtUlDDhVOqoPitO0wcPDnLEyF7gyHoGvIRK0e5t6eWfE5Gc1OK2lAyPOqDkONS+7NZaEVMAeaohCyQejP9yYSScQaZiTB8gCh2MkOMx0kizw52SL42PXH7YMONwfIDsvxJISnMQ5I2zDwXNY+f/cgEjWb6k6kMt49svTEytXxQ6yw4lgwKrPdyFT02qk4z/ZnD2HA1VsKE7SB8TdbxeFTx4M2HYrPka9LK9mQrPXe/aww315DgEBwGDokoSgGhPmgw5qRSjYwEh0R5kGiUiglNxiiREKKP4dXInnInUFOpA13ZSVeHJZ0wLiGYVI0BWnG115Mx1gY7NVqtab/Jv8ixTe4ixN14gTWVZLg33NY9/siMBQYxd/wxAhK49uCXRwGgzPTOhLAyhJhPO7od19kdSBVbo3FoHeRFKmuOHwN6ctvoZ8eSvdnPDTVlvzZoklHb1m/AvYebGsrWjOcAnWASMxsAvM7c7XabA+DuwICPHHdZdohvukY/1S2kH8JJnrA0gDIWpWSJbHp+dvpagIGxIFiIHQkEcIrI3Bf9TKCobLJWJ0FWtwBRHN++EJh3KDA+7WwUOJR2NfoFBlA/2bu72VOGnjzSzU68ILpe4BqExeKUghHJNheTBABH6FbXM3jxbcT0vlGPFTXhoRGzQSuIEd+Ta22UPghXuvRRGCOzPIE/A16NBgon3ZqCGLcRpETZP3jM7VjU3sqEDXka9Yxj8zGwXUu8+o/z87MP21zR2/FJW9zZJpl+clkTl1oRlwsrIWY9LCGYsRGqI0y/3VrWkh1taGxO162QLrOHNIWn2PAQCulhlY4pnE2dGyDPLiB4pzMLXEkYze24ocogFgLNEMY2Ac8kjPNPcD/jN+T94gTEc3fUjTtZR7k9a8t/JQ3UsgZgmpbKPOJVIXeaRMNIn/Hjd3xdjCGIfv/+dPT23S+/fIxvPluqKjqkjJFMJpnAFfq76vVX26pJ24ba3docPPpnC1W21PPNEOhhnsoMG8VRw0J9lJQmSRmPk1KJlLa5FoHWbSZTLd7sFcKaZClmWlOOsXBHz8Q6Bymzj6UC01pJd2XJQM9OiiHVGGYwSNbp0/mbyzMOMjRVbao6KGRn0FQ7oJDUgDKmo5en58cJUmO0TblaguuJ74IXY4laeGsvlwDc9wsf/bw1+jctf9YKTA/MCaWV64Ij9VhkgJOJwFz87cMrDE1acWjCycoE+hsyZjoGPTMxDvvlw6uzJqjsKswmaF3O21HcIO3ChjTRosRSlwWUl5QnFG1RuxVtRQEegLkhfKlvIdTLieQmIRUJqV1F0dUigYKgtmJfJJErkoFFqcsD63/lTdqY4bwuJ50V3BLSOsvrVtIqk07hlLFqW2Q/5lCK8Sre4pisggBXjliobBoroxfLppqFGWcTZrgdSm+sfwQ0j8MJuFzXFTpkMAUzMVs75BerUaD9VWQpJZgs0dcJJviS+kvnK89DTrO71AKb64P4o8kF+4V6AexEGyQVos1Z1dt4NTZMrd4pcRoSmTilBSdTzlBhaLTbNdDI0S4ARR4q5AjUTeXf1P1N14iHBD/Big4Lvk5yF5yXooNCEQP+1LU0IcKsipXXIPbrnettDTqyAdsb3uSjeevqLvPWNuat7eG8+zxvfZd56xvz1vdv3nqb593ZZd7GxryNPZy3xvPu7TLvzsa8O3s4b8Y1fSdc627Mu7uH82ZcM3bCtd7GvHt7OO8+haAGBM5qu2ziyWxS7/cM3UzcLqtRPhiT0bhJWxeuZUWhwLRy7A7RVgyac86o+NAT9xItaX3HZqCEa29SjOH+p+OGzcyWSrkb+tC1zXCeZiAD7WJPzgnjh1ZClxOq+KExxw1A1D5wvNNNJY1iEMlg15xiJiaXUuLwhX7EmbDNrGycdmoKozgKvjlVOW+HAsTpoih2x5A/V3TrGprM1PYjLRoGt0mc1qTNnZYn9BtS4rZfhixKn9TjgmS34znGc8tHcGo75YEueQjr1EGx7RpNY5BwUf09sVCrwEK9Egs1bvtl+I2ci2NfUPQYp17/8uFs+DtimlGBaZ1KTDO47VNgWrcC03qVmNbltk+Baf0KTBtUYlqf2z4BpmkVzIJWySxQQg7aPgWmVTAEWiVDoGnc9ikwrYIh0CoZAs3gtl8ocxRPVToKgRVJh4JPP3XUQeqUyXa458sZVnMS+GHI5+foUFdyRNEUHVWTR8fo2CCedeazrKE8cxrykcV7PBKGxztwY8ifCU7xRrZrhYp4Jc9S0NnAZ/2TEzoX+EzX6BsexfjhBJ7bJFL5s37yhFPxHKUifj77cHZOJ/9qlmtHI9ySUZbrevb8sq61Zr4zFSsv8B0Hz4wsA//OHDuYT5uvHPDTxXly2uofEB3yHtdAU/FsW+tP4h98gPm/S0jVdVVLKOH8+JjE0cV7UftHp/1M+JPJaml6k3V9SGHF2PImC3GF59tay4UZWuNrcXr0UkytCUglsH1h4QkXeVIUBuJZUWtsmRHvW/U5hOMD7XjYHC++/MRHltOjzP/8Q3okbgPapu8YnYK4ff30l3j89JfYcvrra0fx+uogPf1FpyUfO4v3CCFV0/p8qK+M0O7nyNS2ZuQG9k0HyTqDHu7hdw29OShhfSYfTKIJAxj+sUX/dLfoe5pKx2K7vVRTs5uNm1ZQo9kZtKeTYywdxP/agRo9TdzE+ySyHad2oGG8mZXywOBHlsgRH1ciERI1KbF0mpFmWS8Rq5RWXqwSSqC+ZZQ2WC/J8M/OcFPIEEfiA1+lPVk6jJgdYODjPBaw4wrvXDPDjKE8u9yjVerrm9m70/YfC/V7WqgOnzIfDAoLpf6xUL+nheqSh9JXO4WF0v5YqN/RQvX5j+r6/W/aiG7v4VZ0nzLojUH7W7aiwbncw5n3eObat2xGq+093I7uD3jmxrdsR6vtPdyQHqg88+63bEir7T3ckh4wwg3637Ilrbb3cFN6wAinttvfsiuttvdwX3rQk1PfCeP6han393DqAzn1nUBuUJj6YP+mDn6InPtOMKe2C6dq2/s4eV1OfjdfruDMqeo+Tl5inboT1qkFf07V9nHyEu3UndBOLbh0qr6Pk5d4p+6Ed2rBq1P30KsDXZWT3w3wCo6d2tnHyUvAU3cDvIJvp3b3cfIS8LTdAK/g3am9fZy8BDxtN8Ar+Hdqfx8nLwFP2w3wCh6euo8eniYBT9sJ8LSCh6fto4enScDTdgI8reDhafvo4WkS8PTdEnYFD0/bRw9Pk4Cn7wR4WsHD0/bRw9Mk4Ok7AZ5W8PC0ffTwdAl4+m6AV/DwtH308HQJePpugFfw8LR99PB0CXjGboBX8PC0ffTwdAl4xm6AV/DwtH308HQJeMZugFfw8LR99PAMCXjGToCnFzw8fR89PEMCnrET4OkFD0/fRw/P6KtNPKDXUDtar6lpxtP6+6Z/9pnsT7+3v5nYiz9p+vTH3zT9K3BtL/6o6dMff9X0L8C1/fizpk9//F3TvwLX/kf+sOlfnmsVrIFWyRrQawWg7ZPgWgVroFWyBvTSTGj7FLimV7AGeiVrQEOHtk+CaxWsgV7JGtCToe2T4FoFa6BXsgb0x07Q9klwrYI10CtZA0qEQNsnwbUK1kCvZA30Prd9ClwzKlgDo5I1MNrc9klwrYI1MCpZA0Pjtk+CaxWsgVHJGhgGt30SXKtgDYxK1sDoctsnwbUK1sCoZA2MPrd9ClzrVLAGnUrWoNPmtk+CaxWsQaeSNeho3PZJcK2CNehUsgYdg9s+Ca5VsAadStag0+W2T4JrFaxBp5I16PS57VPgWreCNehWsgbdNrd9ElyrYA26laxBV+O2T4JrFaxBt5I16Brc9klwrYI16FayBt0ut30SXKtgDbqVrEG3z22fAtd6FaxBr5I16LW57ZPgWgVr0KtkDXoat03eNC3f6lb2ysepPZuJVmtuR8IsK2/v+lMlCMV4+70DqrcltJk+0bq6oljjabc96wi13e4axgGe7XuE8kGj0XiUOr0ksEtvZu/KF7MvV2PBVUHFW2p8YUXit/g44VFcqdT3LHzDIr5dEiu6mxEVCxKzwHfFzc/vXv3H69PRudbp3gyxpmja/eomoXp8jK+IHFmeOXas6c21fFk7l/Ia+77TjKviHIlP/JZF+UpFN/dSxXDhr5ypmOBb3/EFlSKtrRxivWt8hbzWep0Sk2WVM8PPDfvn8zevtddy4I3SgWNhVm2aHTq148vx4Gk2sxFI7bFkZXIttJ3VCAS+cCPwl1Z6kf+MS6NigCosEJ30tN2lU1wa/GfPiHnZS/jPWga2Fzned7XDKxaFa26HddFs4Be+gVLIuYga3KEVjmvY40tJ64f1YUr0C882V9kWSyNP7akZWVjRzQ4zpdvMwI4WrhXZk3jFkP1Yq2qKlSdzxAJr6ZhYynFriVxePC4Bdw+EkyK5b+jt/+swR89fRi0QgZUHAsOF3AN/Kivezh1+t+ccS8rBg+mOhRW6xaWhpGQc4DKvrDgRrmsqdjgKfdeq1cXz5/DM6fGx5d0dH9+ZwcgPa4c5MTqsp82HKU1YKknyt/TitsUC4ZV1fqkiedmyHWaJf0mX6pfbWqmoEOC6ZjN/JaC3vubGw6PcaCeFGkBk5VhgFqLRDBYFmVo7nDt4cwSweVj/90K/VPC3dY5bbKPAGrKtN95NetJ7ODtU0bnT7TVV7XH1wcL3CrLgIOZiqv0XjDEx68PV2EVQd3KVm7Nog/ZAyuq/A9B8XtlBWl+cCziH7qjXkTOhgs0bAITUqFayd2cHvofv/JRSiRg980QehWrPcQJ1fLUt4k9WsGhm3JovfokLyeJYXppYG3HK8nTztyt8n28TXw18DfL+V/mr7V2LH8WnK7hMv/zn5U38Nts3Hy77KbVcWcXae01RxaUZ3oqXTZ5wo66IC9O1QDWtAGYdCjPMmocLGCsALJaT/9wftUF1zJtr8f/+z/+lZwGrW675KzzgL3BTXPin4nMo+BXIVILawfdsrnnpwS0Em9bo9GPbVr72NGpUrzD7xmSyC1g8uS9q+GTHMUH1J8ulWECLVuS38BONz701TQklqCnfy4yFw+5lsfhYeiaW7cD1upJ0i2vxHSWiIm6A06Op7YpnMIKTE9G+aYob25OXwCmJr1F17x9OQBBvaKQprWDleTABelN0iCb65uP52U9v3r0bvTy9fPXnm3qTxQ5f8XxzdBO/5PkmfctzSgvLeNKjbvi1zyAv/ILqfNFvYJb1EIETSK/hNQOL3nvaFB7WDEypYXEWEBQQhbMg8MEFAjxEDkllwLco+1FcfhDXnjDa91IF2plPiaXB3o9MufJMU2qPTDkz01OCBUZwM4hsxCxsP7ahc1qU/Sh+OFWAR/dvaXlY6jRGmleXp+HOfCso2MIMR2AAEq/r366QA/e1iWMvl+vj48j3R67prUdmMF8h9IT1a24Zgw9qKFAALa2xuvX7+Ccp3bYsOFOubIRP51YInPyhBgL4s0Mz+VMWiKfWeDUfmSEY/mhkff6ullnipgCX/zDzcOC2lIRNQcj5LAWaGRF5hCaX2NS11ttcVXUk3Soj/V0tVccNmuSfkhq64NfAGNmLBZCEkCZ9d3U9Z8pLaP9p+2jRC7dMII5GKCGZ4wM6M7VRU8yaYlQHfEfTkDexClDduAKyAtBV4+rU3XYHCwt1NV0WL91uUfHfc3cVCRtAXrzAb6Pk22R059ubngW19qq0vs6sQPwC8AVqXgiz6nMVVPomi78zrwmiSFER2mWFxoQOulsKG+3aC+BQDZEaxOQulkAs1Q5CCzGpCh/oKie/wKLQ4Omt4GE954V+w+Cke7vdSZbeMfrK5ISQZzwkFEtFUYCzAuBgPZiTyFkLNePdZuecd/2AA/kL29iRXGZXgVlU3+ycYVb+Tnvj9wwbM3fqqVeWgNlHfB+/eCmNbotCFXJjjjPVoSey2gFCeM69YO25uY5fCTZAR1FtG0azo6JcA6iCNxNGYVakMwjy8fKvo4v3vY7ikgMV1r7/7fu6MgGbESGyFW9/SW9nVTLW7qQDjtgEV6R2qEQAv6BT5DkdVu2FMQqWO+OX3FtTxe17/Vu1q4BsQTMHUzFK2Kf/0FXOAk4a0GUijsAamyF6Mljs15tjiARSFPll4biSF1UUfemUxQgYl7RACwddwO6syQ9xVyGHcGBpVTCVMkzPB2IyZRAFa9CfeO55+VFmYDWBd3d2aINjLBRuDtFBZtmpPEXtsL7R1XpYWpOodkjjpn45OMYR4Gv4RzKQPRHP4zFcKUo6tuthyahL+qQ9FCXbJyNlmcclknT4bAJLDbBwmJE3tV5OIPPsxwm0iwTyj89ImO3F+TxMwI3mazAQBAcFaaIqfgJXE2zS0rQDctOnvwKAgU+D695v0duKpbBSoZOxD4G9zFXlqJGMBBYOg4zcHawWIF0YYRVARE0pi0jX88lmmwE/ltqESimPCrp6uJkDzDIKyy3oWhm3vivRxtKcJLMpN7VTYUhOnJKfR1UrkFnEDKA3c+xJ1JoFlkXojrrHQSbwwTWXOWI0fzNaBaxdzNqkm3QYpuhSgvtJVROpwjGzSFwurBy1yHKXETbQuzxCHltoRnY4sy2seWJTem6J8XcQrZUia8o4g2U89HYvZkfVLlz5o6zP9sVM326d0xlYSWNTwdH8jE0MmU7E/7Imx8eYZhpNzKU5saN1Lb/yyFIqhDECnAShbCtKfzOnQ2y35/K2sXmbK1TC85TlKlzUarWEHr0MnIpl4Ne6OMIfzzZkbyMjl/nKRCEaj0YrCGwxS5BLQvF9CBVWy1p9G/pgE8fykFvoKdMKJDLH+hj5K5AGrgqPHSh2PtzkEso0c4BzlAUWoWSN+J3l2PAKtx9gicBr0IzrMqZlxllL3p3eyNKpy5DhcX7lZP1lBnooiQnWyDNdUJaNiuykY8u4XPvtuKHKaqR5A+hHJqkOemCeNJlxvXZU3zQNEnG258EOqcPYDALbCqoC1qPFfLIyr21Z6jyHyk0rxwfxI79eGX7TvlL/eCT5W1rG1Stxc0qQ9fv/3f6+Xrlt8D3M/fDV+bufxD3l8saIg7+C/YClHa/FMnoww1yoBLE1wsEF5m4d27PqGY9DwQthrQ6+BUTHAUTZ8J38jr9jdOXU/y6+czDza4YT2wZ/D+R3224Qp2kKG0HJZbkHNB4P9IFqKEq7PxkY2mzrHlDasbD9k97i4vL0Egn+gAsc1d+N7kcg1rUDcrEpl29zqMHBsM+7DN+jFb8zPSy4SzabQ4xcO/wF62SFtu8pTO8T58diJUoTJmnCDaOYpqwaJkhYuNUNxhk17wjiixtK53C2RF7FGkA3Q1DWyAZoskwvZOcTgACdTg/UGrUKldbxQ07zyzILruX6wRoLlB2JGpUaowpUwpyj7ERC09sY0twv7AkaYowIOA1Aw714r2BJMujKxcXyfXVd6z/el/J/NJXXlmOPSZjAZoP3kpQiAwMcuIo49dCHtmYze2IjfLA5B9xIBNr27uCGJVkDlHjPzPSQI+EKJjtEsTaz2yBYzAyzmJJfwBxyBUDq76nEsjkGt53lZaBhiIQfwKNEYNwRevO1g83CGBg7IYuXVlgnnyJNn08CUzo3uYxkUumuSIudHRA6lJUWFvbCUQKe9rN1vygDfG+uKRNPJdZCJU8LZn+ryOQXb+BgR9TUOLkHpqPWr5fZm5LNrmPyQuWeFAQPJFlsCcG3SjciS4lFCzD083QHC1kkgwSKuIDUwxEnKI/WJC+vLk/zW2YZasBcgL87mRbwAxtcOgh/cTUdq0WpAbQ3Yx8iMXD6WiA96PwppcTeouGjcnbmQibEeV8WaUf3Pimk6VD8RrW5iemRXz4yE91QEO7/so44RicqxBzzAZ8RR38gwiCO30+BNG4wF6nR8m3se5SuldyDXwUedsjmKkubxpkQ0NTPIIH34aRJlQ/xJ69AU6ybUj5tD/DJa5ZT2nQ1NjbkMmyhN/N8ImDAhcG8BSfkQXbvUa1pzULgWuatPXkapXQ5dkU3AldETP0k70NpQYTBY4Zcy4Rnc/KEq7S3dUzt6Sp+PK7fjzz/0ry1JPZimgBQxBMUZAQ2jQJQaMXZKzMCUMN8HyWpSomBpKwJ0oFXFAjRwMnKcL1LthtoA4FzjP5z3yJZbJXTixPprGoxNq9Q3RDpJYCDxG+YhxJqMcpLg0i2A8eBuMuV6Wc2DNHFb5smY+fBoSnZNrjG1wfH1qni6ErXFUY3M13bsSGkPsR8JpgNh7ABoiq45TiUhI8tf7rvX0ous9c2tgjM10sLhb/06bIgTQ/ls9/vNdV+knr+ebl6708tZzPzjNX3ItrSMUPmW8xOO2QFwz0bf4auHCI6FntcUbz862o6t4qG4x68HXCN/BlmGH4rjhK9Rc8DL5ES2Cs8lVS2Lu/8OSUmk61CTAsT/M183IORzHPZtieDgihB2W6TNDC3jgXcDtaxCZwsMHk4ze1QS0+NbX251MCDKU/HrKNNYRY6cJKNBwxTPCoPq8QnGCBcAZsONrWUHtlHOWPSXRQxOaR1Cw1aDbz0OCJ6p4mFLf09s5TeBDALoiTw6iZRk7a7ZtY9kJSGE/e+wvqQ8Yd1WqYbS5iHC4ZsIYCDdYM1/q7EvICTcKtkzypJxwGvLIHTNc/baoNK2fxFQHBpbesCSFjsMSwXN8xVgC8EY/97GB3DhMFaNIV7LJ6jVpjR3+EhrbKHYMrHGtHCXoXRtWicSHnka7XnrnJfh4haXiQhpYvAc49y/MPy0X8j4WQ56iUzTum75iQh77G6dQ2g6irxLmDmktzq4yslZMuYi761Y65BsMAa0/kJ+g08RfB5atvCOdouOFoi8q1zAd3GDRnSdTqTbteEkG5qdLu9cfmxvs2uuaBu8yYd6DNUOtEHH1rOTZeZbxm/j6bW5xUgGkjMFLf3wCaAJaWpxU4LHYE5Stze49Ljdx5iyxLjrey5u7xvG4M9U/MxAsnuW+GWFbm/IQIxKi2XwM7YQwQKcujNYI0e2MRZTS0mB6EkwhwE+k7IDtRSnpzxfK+19AEdWv6shV4riAHW9sqdArN8Qk2Ah3+7QnZeHzQ2GCYB4et8a8RqSTmDGjuUt3VMHMyXK2jCah87pl+kCjHspEFJBhcyx8Eu3r75eAzh9B2W9saNmHt5ggj3+SFa5IM3LTx4I88D5FJw/Mxhcv4nFvJ0L4/VhAuA1ykDl/auqV2JLV0j/tKhL9k8Tg2TdbjxrbXb5dchSM9epz1BLLaW7ZHL9SXrgHJRe84sfc7nk4rDbqIXElr1dJp00ulIei8cjHJcY3lztPupeV15ID3Ikb/cW56mdFptpfNS1BA5cIRMhbasXGtq47kKow8wKAN4UrxOu6kaoHk9+NT/e5qHDzEdOiGQpCzG6FRlRHZsLcw7218FtPSmmJtLGDjWE08dbkrz01k2pChVdQYqwocu6T7+ugJpQcGAy/gUiBcgBlt5eBwStNGLOLqT6X1wK+R0WzFLyW/MZmnooJMPfOJ4lvYPgd03pXvHN3yE0/OZYLh0AE3pHO9bgckNOpVFPkt8zOeGQ8Ok/PoC1MDF9HOSbkL9YHLwWPAA+CBMrxu7uHS+B8/F3BzdrJY3TSzxQtem/r0H13z4/zNc1vjqLfx6d9NkisB8Uxjt1sV7PCIqTpO0SpLaiN1D3NVnig2giF5Tt/sMBykbMj1wR6Etj6bBo3H9O3KG71HN0zgndmAj378FM/FM5pyQxQ8tCqJzKxFYLm6RgX/sBxGI2jEyYMvMmdLm9FFIcizYnLxMhci0iU165YcSk8FXvCkeC7jZMAtDnCAgIjY4AWPN2D/NpMGYGqXhucuJmkP4JM0hUzBg400Qp5a/TKxFVmLjo1VgYMwlRhu4LhDxWROT0xImJTFvsucDIHSKjxAMWa0oicoE5dk63BXn7S93GYUKpU9k0l+m9eF5EIZbwcTGKeLBUV5WeQhILmfcj05umHyOZ7aCxmj+Huk+lEdBMNMSySWQ3jll4eTOO1nVlkwiSbvZQrsZQ0oAzILJQo/zs9N3o4s/n348uzgWVzWJ+dkPcLINPCF6xe5CjWCcftABEHLWP9ODfAyafj1GcWMdBI0GAAzknwXUVHzv10bPW+p5dwwyKPU2PgLqmQE4BDgreXhGioKLMeDB/weFdsE857AAAA=="""
WAVE4_PATCH_SHA256 = "8fd9de6b6e2a41b84e73835530b3018bf037e621a6110737bbdeb26139021bab"
WAVE4_PATCH_GZIP_B64 = """H4sIAAAAAAACCu1be3fbNpb/358C8Z40VEXRIkW9k05ebjeT5nFst8kejw8NkZDEEUUyBCXb03rPfIj5hPtJ9l4ApEiKUuKJp+3OWZ+mtAngArjPH+4FPX86Ja3WzE8JPZoF7sqjR+yaLuOA8aMJC925kXAy2dl04IceuyZuhw3brm0YQ9O0aHtIzHa7Z9sHrVZrD92DZrO5j/bTp6RldnpdvU+a6gmvpiFZUj/UGqT1HTlhfBWkj7WGTp5H14+9m5Dw1BuNWJJEyWh0jI/vviO/HJDsJ2ApiZNowsgT8itPo3hEVh3r12IX/FkYNE1DJ4muuCO6a+V2/PkG162TTzpZuDpZw79oleokdOaMelwn+HA8f0koxzl0EkfqLdBkibNY6wetKlExiKeJ7zGdcJcG8AjTaJETwTXrB807j1vSa4ezTxU6JRqNzZ+3483vR0fkA02WhK1ZckPWNPFpmBLtz6/OSJO41J0zDvyPgxUn6ZyRhNGALFgSssBQMuzbUobi+fUyvD2oysljbuQxIa6KoO5dSJ9jdFk0dxXLZvGNPxVEgJsw+A0YRhKF/t+YJloVd4fKQob3YyHTKCEO8UPSNgw/ZWD/9baxm+e/m3F8vVlUDKIog9vPiEMIw7Y7utkGaYhfzHsQB1hfCtOsZvN4lRrkRRTyFO0voCuYn0RglLhJMMTQZQFHwaEVeuDZWQKulHGjRAz4DP/8ECRLoumUgz9MIzHEna/CBVn6nhcwwiNp2p4TMFwjveHkv63hwKi40og7MAKcKdJtEk0T3D0iVoN8S+yG4GvPHm8kg6O4H84C5rg0pq6f3sBobUI5g/HZaPjVbCihjMu6iY4Ld3luCgHCf+2Le3TfZr1u4i7/Kf007+SysXeZOfWe+u6K2e0JL2F3+/fiJaqeQq261mF8qUT+RVK5u2S+UjpbEqpIaaekvBo4xhP3SEZTrl4ZcXq9wU317QqY0WnH6/d6AMz69mAwtGuB2Q4KJXS2ow+qltXTe6QJ/we/9/TpATFSmszQyJdOvw1/Us9LGOcOh12Sng1hqwlu6F3IlAtrYYNH+Jwm+GCzJQtB4jFNUj/1oxBeTm6Eg4oTNvWDgIAmQRdoEpTUskAPwY9xN0oKjmXaseQrDsoaBNGVJNbpkeeAU7yVi1SgR0JTwJwHTYNdgwKHxFCrMWjgz0JiE2MywA3lOnx+AS7tAP1p6/5+BL1Z4ADHHFj5iNyc+xek+YRcw9Mg5EfBrxGZgVISl/mBFh5Z3V6DXBN4YJhAQzDufVlCyv2urQ9Azv3uQB+iA3n984cT5+W7t8cjMWEhWohggFH1PL1AR65jNAGukVSKzuMYcgDwu2kgJTv1E54eqPCkRjZNSRNHcqL5KSfRVQjTClJX4AWuEh/JFdTDZR7YZ8bHxdrBLhKjKG0bwxNDn3jFrn2ekskK1pUwEiLARfjqkRZMveI0aBgHLaBVYryWYxgMVsh80xrkzCendMkyZY4DGqK7gtUJOlHiz/wQ4LFm9shr/3mmm80dGgnTN+80vU7AddOl72YrmNykQP+JIFO2jW9tMa1RNRlfovgAjRiYUxCrQBY+F7QkM4Vg+TxKxJlqGYOIwogEUTgTjOQsWTNit4e9jREm6hjx4uyZVNTTlM4YsVzDHJH3lHNigusFacA61hbRwP7Jh2cn7wm4dzD1GxScTtwI/DEHacNCQiYNR/pqeBMBI2BvfD4NWmnCmOQrAzgResRjgT9hwF0Gqvfy1c/HJz8cn5IprF5ISRASLr4F3IVNS+9C6ASAlk6u5j5grgVjseSSZHsL1tZCpVybxItS0B4kJa2mb+pmF82mDw7SrtoN/hjg6eiSGACTSOwo/dcrbeDHYqcc53Z0yGKWjINZ+1S0izAmw11Ng15uWamWonYc4BlRRXcjYTPojFr2MH48+G5cfD2BwQ+Tx3blNc72cPq4Y1V6w9YfJt5j24b3ct07HDAYxrnZ6wzsi7FAs1KvzgGAoJpd7B8M75LzTg+HisFXNIllnKEBaswkAdVxKXoiubrAk8wQsoEFgic7j51PF+P6Zks0Ly7GSvbDtoiM/aE69lYlvxkvuNURw4tC3p5IdLTzjkrYW/0kn+VyhWwvFAKvUuqpLkUhI7WD7LhwJqxAOW+wH3TD0jcCkg9nKbilPZ5fnRaW0TqbEHj00E2p7xk3ik2Ddk8EloFpbgcW/OHzRI0ewuAEltwdb+GtTJy+JwdBDDW4GGS2cVRfJx1zXCUIU6oeWySBYOggSS6VarMHE7MnQhXHNatQnk6cZzSpiI0tCh0908eaaTdhoEykWbuMHJKMS0TyOLBZjiTA54GyTruvKxWwxjv2sQkL3xKEadFUA9VSSymwuKOrBSHV8c79qLAmV6N0zF2n1EgjYxZEExpkptTVhb2NN1Q+Edlj36ieGGUVRi2yUULVhm2RiLKGIHFhkc/Ozt6eOKcv3p0cO2+PP56BxhVfFZRwQhMB10kbraOlyCM8klEL0BqPpumSXosjeearwDFtzONC9JfM2zu4LLxaIrldSVMHfrWn338/aOPPuCrLlh9Oq5ZoCWn1xtmO3zz7OBJc6rT7HeQSPIdFLr376awA98BpiHM4JzSLkTJuaGgzrVXoQ6xfNgRCwEjJaAIBN4FzAQxRrehQFHYMQ4ibHni0IHIXQBLYnfgsMcgxbBxxWooHLdRkBASCL4gh4Wx4Dqd/AR/lugBwRRMEHnQSMCV8ARRFSgPOLJQXp2thDqsFmGAJuCxlYqmITwRci1E2EwZwgKT+knkK3KnMC5hlzIRP3EZ7OdKTAO5LCVXAWz3Bez90GGuf+8gtA05VAM3wBFJ7Vi8jFcAoUl0GXQxznfbQ3KEuX4tkyD8PWGqJo+YqJCN3MLQwAsFzCDvZuYMSsBneO7CJvwrYxL8dsOmY4EZR4mZ7uEfif2B0U9d3IPuCatwr/pEMs0yhYKbVlxCnlmF/WJwTfz3Oif8f5/x2OKdjWaaI4JY12Bjoe+fUdE6Oz5S2QShWsV80nL5+9X705cgm/j+DbN5LZPO51Ooy8kql7u02lVJlE6/XnnYNo98Zumb/8ylVNXpnOlW1C4DaxZpREx6WEFu8mhAXCz7k/dlH5/RNvzsi34BTBD/jh26w8hi6yAfaoSTr8GW/i3nZwwZsXg58/uO7F69FjRsGWV3kypEs6MZC08n//P0fMrE0Y9GSYfzHJI2AHCFrCTCXpTd+OH7zs5ERxqRMRhdrNE0k+320SirRR9SFEbTlQYjwIEoNcgZTwLayPBHHHJhMV/FIUsNFcARKG8sK6E2E6boAM4oMCxicrCEagmOeiv4TASBnIr2HSUZ/Nk8lNeHjZPYPUJncBCqIc/LuwykYxsufXpy9evfWef5fZ8en+c56IhhMQ5FvdTx/rYUjWX7wxFOUT7CvyoqEBvRxRHIWXBlm/eXsL8vQbsmWkWL1VlZ7JNiVp65F5kuFHNiWJJdX26f+Nab5cNet7Yw2OWHpKgkReF6+BaqXZEkXwGGK2TzBN0nubyyJjtBApwHIOXeGNIOoog5EgOacoW4gLA0BxXsskcn7yQqrjcBWYNQGO8q9OkKoWtnPbnj3LsYVP4Y/sb4kPAiIsuKVnzwh7axVOS/YFsEtqdB/q7x+OV2VDzDAybgLWMxyFWh24081LeDmtZ36AKHgVib8VekCdBnFGHkrAM5CGqDHUbCGl8AeLyglO9VtCPKfqgXLqZKW1N2ruQ9ULh1J7pIEPtoDWqYwgTQBsWLymwP8kGTz3DlNwSLRT6her8VUp3DOkuDWtDqI1cyOgmr+Mg5KnXJvClxHP2B51fIdixM/TIMQ/My5dDQXxGq9JCJB2hK+I9Nh8BBvCAvx7OUJJ7SpfjX3kVNhoZUdfKomUUvz3UKr3YiIBktaKcol4PzUiaU7FNGx35NgrJ4lKJ3LPdDuUhxtGR5P0TfWFDYAKm5IYelCbOuaaMI50SmWIUCesgihyvay4rFdkWgYG1In7NPKR7CTF+0fcVUWEfNPWF4ZEQdjopULIVGYRfmCiwX1YXSJBYedTeSybF2X27WCAi6GHRVp+VkBhmiXYs1OoeJ/2RgjJ9CX8OohWHnKNCqvTNwykkWHjK8QC1Is3kRTFVPQ7e2rPuDPf5xTrAtqbuDH8c1olEaRs6ThjQN7WmElkjcuZE80ssy91d99+YazYKp0rN8V90A6Q1s3rf2GVzjqjCqXUMTpZoSxoPASOTaq3Hep8a5qRLW+/0Mgq/nVS3HaEpTuk07wsZCPtXxEDbyggRV5ITOdrNUTInHRHnMqc08OjLl8zmNFcK5ecBdIls1Ty8r7emZnlbp+TUG+UblWgqSBBbDaMjsqaxRLw6Mfh57nletigguYdPkWf3Py31xnHfmeXtN7sae3UIXBEA9/Npy+rPZ+TchpAsfutIY5v1N37u7p3tzqjjz9AvIX483dC3FDQBq8hlZhTJ2q4eiVUqZOTFBPzYQzC5aM8Q84r36zkVajcomogC5AkF8EOxpGtHCixAHczbRffy1CCvxRtjEaHYczP2QaJilp+kDbviFyqOBTTaiSoC2HLr+UV3B7WCbWKOzqFu+ANOtYWB6zi6HlXjXcrXQosLoyQYGDdQoh5VFoUTdHbzeJE0C8Pp2FEU/BlWP+s8XjwN9cW5vW3+jIcsmauAUs7Mc25anItge/sysVrdmV4T+kexWk7uJJGwWbzSfgrnLTqaCnLj/hzqset9QdWFY7rpyQzej8ni7Z7rQB+jXtnq1b5r+zS96QT+/Lg2+qAv8eDnzjeuRnAb+t+y5w8w/mvUVcESmfEbnEK7ofyFNyDaeeyw+XeNRoLelfAU9fnoObkt7GD/F5calu/bdFPb9ptrt93RRlFTjeElhTyqvOMYpZiCrw/uyjscTEBcj90S+PGoYbrcJUq3oLN4g42+p/W9sfIg9LUod9eqCJWXQ1WieHq3BCA7w17ZFJQl0m7k4DycOiU5TjH2g4FRzbk5Q7V3461w6Pjg5BwQ8xg7VcYVILGwm2iRoBCg/zNtESDxGHRceZkSyLpn6Ccp8vna0g2i1WyK24UZhi8kw7NNaYxAH17xvtw8YX9C9e7xQDmp8ZcIc7lXX0hOyKkj7cV/A7zJVAJ1bd4h6UV1e6U1S/nboRcXEE2T/i0V/aj1BV3v70I7mKVgEYQQJuAcvN7uqNSPn8GFHvJU1pKb+BaT5Rr07YX5mbYtbuhoRR2Hp2+uLVK5ErlRpAyRTGBuTwp5Bdx9AVuJz3K9FzQQQU2pNDMmFTdHS+8AdoTlRdfBPn6EQlEjEHRTEDQG8IW7Y8yufKulVeqY332ro7rft241qa8riNXS7kH9MwA4BO7oUz5y+P/Q5M7C64g6d9BxkKuqQ1il6/oCM7Aohl28D902jJNNNpm9YuFdsxvAsj1HDLaQ/sOw63nfawl8/fc2yrfUcKbRiNic67jQLbGI3ePPtYGSySmJmPP1XZHZD1KgH9maFKZtl/tO8uCDTkUdISMVElO1ElQGv+/OpMpJJyalOakCm7Aje0pO4coi/X5VckKf41y9VI6K244rlEqSccCCcsT8dI/dhVrgHDCVmyVanJX6siTXvgDm1rahjt4WTQaU92Fmk2A7fqM5smVPgBRFtM6ODT6uSY8YcYLRgOLDWfPMRzgPEPtNShAJBPz5xnL/S6bso0X8jk2eRG1leUAY5U6Vnc7t2qNDfNneTkCRv4X7jurNEA6dzk6UFxq7VRSDVyxkKj/nsIrIWLXB9EH1zTnBbX8wSPLM0wuyErE4OJcDE7yYlrQZgrLVcFW7KYsIFgpbTdTmrqdrI6YBr1jKn7dK32e5EMwcEZaurAAUpgtsXaQP45Cy0AaNuovF1nb2OJ7fTdlBW2y79kqf20RR4CP5cD2/H1Sr7+3c1iY7uba3f8hd3Xn+2e86j5hTza3bHKvD2TZlzdT2xzMWn/nNn1pN08kQIiO7f4+SXn38U16r9Q3Pm5kTyQYPGyzpWKOH0EgR/sq+RMKw3KnXp2r9efdA3DG1i0123XutPq0JJDrTaiSx12+ggh8JF9kVa1TkdhPiERUW8CKKC+axQ4oE4i25yqCqL+m1EEIoCcSqPV0chYhVcJjfOTxfbXW4Ue/wtn9QMiVT8AAA=="""
WAVE8_PATCH_SHA256 = "03746e114f8228f5aebfe43399b29e179c17c805f084fafd754f9d0cc878fb0e"
WAVE8_PATCH_GZIP_B64 = """H4sIAAAAAAACCuU72W7bSLbv/oqKBvGQsURrs6wlSXc2BD3dnaQT9wIYHrpEliSOKZLmIsmTBLgfcb9wvmTOOVVcRTp2OrgD3NGDJbPOUnXq7FW0ncWCdTpLJ2b8eOlaic2PxY6vA1dEx3PhWSsjjNi8cejA8WyxY5Ner8+7E8M4ORkvuhZnvW53NBwedDqdW+geHB0d3Ub7++9Zp9cfT9ojdkTfwwGDZwuPrbnjaTrrPGXvRZS48WNNb7Pn/u6xfeOxKLanUxGGfjidvsKvp0/ZxwOWfvSZ/P35gB0c4Y/jY/Y73wg2ZrbDl54fxY41ZU7kuzwWzOUJTKezFP5axOENs2CcbZ145Scxs1bcWzreksUrkRH7ZWx2WWRxVwCwF4fcig2YXLxiPFxH7DrhXuz8UyAOi/hasP5wuBtPRgwAnQ2PHd+bZcQQyOKe7dg4mTUPIiac5SpmKPhAwB8vZj8O+mzu+tZVxIChzwS3Viz0t+zF2TNDkvoov/DjiphpMBq1YX5upLMnTIMpJBFMqs1gIvRLn5UxhCvWwCsCaMRljwi5ArTm4RUAzJOFgT+1KpGdGuQuzNZcDPpaSlb/zrCDOCzAo1IYq9i3CW7XZocbYT047xr9E3gwyyZ0oX9X4XIdFdlkLBiPWDIa7nNCHNqvqHF67JgN+vWYTixCROx3uzCSj10Z6U6b12PtEJfTZrCMa5C75NZmpakhg+r6oxtQvtD3gIq2t84YuP7gRTFwmU49f1sS98IPmQlawrqGIadYUIFvMb/P95nqloeBmaCYYkO4oMXC1nSDR2YkrMhcjIZgzY9YT4xA0HK2wBEeN0vUBC20Yt4wcVJROWmp5f9xCX+D+d5L4pLdV8o8JRSEjhe73gOtvLDW+fW4IxlcpN5rSnvcWYaOzT6q7Z4avc8MZvAJZ9MBZ8Q+ZtNKx7SP0yP4+VBvtctMIIQYXZiglq+kk+qRDlNWP3Ok4t6gFYciErEZ+xq6onQQZCjdP/jWJ3/ikxEZGOyD2gcfpvgy9l8yDi75t842BKmyZci9xOXw+4ZpK8Ft03bWT0ZDBu4FvcbJiD3X2xk5+EAksPzAAW8UiJDF/pXw2L/+538hIlE0gIcdGL9hGJ4860bGkRh81ncHdk04j0Lr+EqEnnAj9cgI4l0ed+vH08B+OhyPJ0PD6Fv9CV/0awN7A4VSdG+AwRA/GJ1ghMevyRgD/C+/PntzZr58++bVVMolFDEEbQrYIKLOt/sQvaVr7tvplM1vYtFxML46YKQyqis4itGUBEC8FRhljwHLIGqvMPb6C9oVGalRUSEwbz0wMK8+bsOAzbi94Z4F2z6/YeMZEUMiqDoryD0ci6CiVbJYQGrhhzZoB5gnTtD3YPtvIPzGwLy8HuPby8zYOJEzhzkYwBtSoloBKpdhBDzka2ZA4GWBuWvXPQVHWPdY+cbKEIgsMCl7qRtA13lwpB8cKVcMTmAJIKGw2cPg8cnTWfHxHFAeho97o/JjNMyHi8e9bgUaJvUwtB/3hk+zqOTakj3NGAZ7bXYOq7yY1Q/3afg6ahof0Lhc9z4MzVZywPU3AEgeKIeLbJprf6NGgcND2BzHNnYzVv6AlgBZiQCOMzCWQiIFwFNihj3F83t4yOYhB6t4/4IsdVblNAT4mPjIEdBdJfCTthwe9KpTwDm43FOpdLQKFa2RwjjZQ0AMtC+MwadVtNO2lEc9WiFpRj+brX2duIbrKxLjbOn92b6w0vyIzXkkqgKYINZon7XElmaPLgQSCOIt8a1NzI3YN5auP+duqhgoTNCu2W0wJFa7fysMCdIekGLQ1j3/6e2LH82f3r59N63b+X47XchpvvP9+p3ndi62XlchDojCSaoDtm1EBQD5dzzLBb8Fh6to2KcpyLCILhdC+6LEcpobglqttGA0FQTN7YDPIzWG01qk8oxWC5eSKcMGLy21dIGbjmC9EWpqm3V3C/XJFrwrEqM/g7tQHH9rgsNvTbD/rQn2/hRBx94pekMF1a2hJ7FsZ2OEniKKJoE4vf6p0S14KDkKG9tddNVnVlB/cS0hApo+ERnl6j/I1f/D2dv3qf6HVpAzHhFibmYEOM3VPAM8bUtFzTmA4cKwg3YiYcIejadavna81IbIL+NfWF9BoNVR7KEUiEfjFKIvIWYF34OOIrW/ibS/feOTZqscziSVXJz5mjHZXa97QQT6GX0Srpf6FjJeoNHNRTvMRfvm1R9ntZ5lkHlk8g+TBuehlj+o9R5y5cob9np7K0C50xL6F5WNxGlNq85sopxdKuVsEbl3zSnItPYoS2uPqCKpSamWYr3BdKqrySS4nBxt23VPd5RLfakKWPt2qau3P6ay/9PBxOqdQvY/4daw3+9+MftX2I2ZvxrHrH80avd67Ai/Rpj0B8kcdiBMLEiLCfoDFLEf07roWPXhDPYeSijIui3BFqG/Zpevf3rx68tn5uv3P7zsv7ycgZKJHOn8MqM1nWJp2rdN4XEQtH15YUg4+Rjyfd9322nf7Zj9vhKQfYcQ2iGnhvQRcsiF47pQsoqIsn6o3DrxKsTpYGWLOUTewGOyHkiJUVFAeQrsOXc8WcmJnRPF2D3EVKRUWqR9wxmzheXbVN/dRDk9AEECfugsHUj9cT4dIq+kJCGLxQwtTrYtTNDcqRJy9ixy3MQEI9obCP1A7D1E3dx7WKgB8rGjvbFsTjU0Sd+bR8zIr8Vb14xS83gwQe066g2H7X4ftcxZB+6+fuHHWShNKD7Ej8iaIK1zqdIXrN95KQtz2e9IdeP1q59/Zkq/WmmfudK5oQZlKgT2RHWrvc10uuGh6UdaS2k0FJzv3/4OqtXSDSeCxa1Fqf0EE87pVDpPdXOmAjbtw6QzTvclTKfNtErdi7qZqqPe0kvtqFv52TceXztWJ1rxsGA/PFYVa62g3l5ptbtDLn7N2+UnIdhg5ZHcw0orKZNTBViZAnilxBWQ9cbmAswHJ6e1wAPDIDaBW/p3e3i5uTQhpxBNFKRdNWHjaBOmNL4mTDKWBsyShTYRKAARjaNGGpkl34GUgm1ajzL8WxeFILfiSwfwRRoI1kBnfTc66zIddDXjkzE2scajSRsi8a2ehppf0lFrkXAXRkmobaZZwnFNSGQ1DwsondLnHnxpvz97/y77D5Kww3USM0oAIn3vUOuY/ZKeNHGy+zX/hx9ijyh0dnnzqiaMYezKQ1ZOr9C6cm8gegkriYUMZTKsgZmju+Du0qemVZtFPvOTOIBpolvBBpid05NnZHMHzxdCwVAg3FsCke3KcdOTN3A+CVTYC+66kQz52Px4hHxin36raIcJxMJjjZ0o/ByivNvlNvqUHb7Alnz+dDdlL361xcaxRBCHhYHrqGlE9myaRnGWU+rtF1j7bulZ9TTztSvPLouu3RbzZGnyKBJh/ICO8NhT1IPW/qKZJ4QdYSfaFTyKaathpOTAi+RMcV3t8tMxxENWmjV+upX/67jT3NYJMMbYAaoBe+rDnuZNl1Z9454OJ1Gtd22GX3hIgt9SwHRWWTo6qcVF7qbEw0WYhFY48ayeWmZWBHDn5cWRie3wZOQR/jKzX5a58Z1qoDmUU74XuDp5vA8Kre9eGCSGu2BcVM/HlKMq09x3W1mALcEpoUuPVRmi4qh+rFu3BLlDhRE9P9HJXMo7lWXYThRQMxwP6SjzKeTmxVTbYGelE3anmGUHcQc0F7vutnCduQgBgjyf5SY27Bmu7kkPnZxK05XLyi4LlNzdMuQBuDMexEmIHhMvMYDTi/0EC4xmLyY99v8fLwaJK+lP7i0OD5lyZr1qJish93164ymqMvFCmsqg8hRVumj3nrrLYID8rSthY7qmEbbhX5l+aCKi9ulTFRc/amHT6Su8ACK0ohNMYyxeo2D+RoQLF1cHUmoZkCb7mq6XCX4und7WLbt+vZ6+dzhcON88Zpc3sMIde8S2fz+7lKYwBFPY0rlURNYpswLdYJfbSzxhvDyHeI1HlG0I9/h9cWkUCMpnl9K1zzGzAJnFToAHUgsIFUwjDrK7TddIFPZfzvFSxVazXCcIbqbT2PdBsb0bk4fLRN4AuZA1W7d/Shd+upNRm44DIRFjkDvEUbUg2Euz2/vjWSJdM5alytWAVsmCbx9OM9smBpS23jZIeWQ9wLoIIKUzpsPSXg8qn/6oUTyFoA4Fl0cHRqasxEzKxDTQxun052d/gJG+geRg71bUX86R6IUq4KmhYeIxfFYxmgEesIcbEZmQAZpXg75JR/Jpsajp1UtHc99GhXx39kdZoEYUuE5sYlNHa93tcLFVsSBD7AIBPFtpgVssbIHMHnzvflPIm3GNnD2xixuYdQvWXdgZlIdBZ7awH639BjoU/pT8oiM90e9HQ/XMiyR6+yQUvsrRgEBz+7yl3wG9pgl+J7xK0/tOODX913q8BxVEQ9rB3YDnPCSB3g0als/L0CV//AFv+8kmZxJyly0x40C3TFc41ubpCViyF/l4wQNSBFmByrsfMfvbD2fYeMypLXjIFmIrsKazVhCEIqq3LFQG7CpyZMVvqPJiK4EEsfkSQmIO1DO/LM28qWccJp4nwr12cfZYdYq7k/l40J0bxqg3GC9OrcZOcY641yTOh9DTnU567VN2JL/Sevp1kPwMeZZbjQTpoarq1ao0DEOvvEC5Sryrv8p8Tce7EPvYwYpH4oEWm9dXmzb7cGb+8uNv7To28mJXuI5Mzw/X5FhVcA4WJsRnl9/AIsjpIgDdGJRjUM1D7MxueSEJEWAUL9z16tSz288EFEH4woQAvtKcQPMg3gMfve4GWfPtNJVkfoG4l80+XwkyqJUSxa+CfK7a7FBKZ3tN7YtrsyiPRqb4pCyl+7K7InZXmzvzu/pz/Db35bcp85MNpQHqP371hnc2AAE5FjkMjxovoP9SsdkWio9AhOQh0CQctDWGvipitu/9Na6lZ/sJxMEOhRCDbX2m+Z0g9P+hy1Y35IvL0E8CMDnMaxkkjp5xi20JNybbevXTGdrWfZQdDapR3UmRvk7h60HTqqrdPJzO6VaI6+j28dIVp7pPZnDNICUrqger1eHP+uyWrbI92qlnbxud4C3675P+V5S/aQfhJ6pUjQWMKAKMR+M7RoAmPWvw4ZBAlDx3NpE6F/qVcWCx+K8OA7fr2DKhXXr969fomInJk5kEpGorx7aFdydXi2h1ujYhXTsd3dXb3r60u2lgWrzm8pazgx+4MNpDuTT9K5Ujo1enHvuk/2/8Jk3qP+43yzrzrR3nyzdfpdRY+tU5z6Z9rHWedbk8dQiOA7qHXsrmKwMqn7fHfT466RqGdXoyGQwGtfl8FbWU0VcHqXsxGVLzAr5OTtWrXHkbRFWxpi1SfbPNUCygcMGqXE/FqYpfy/UjoR0ufTC0wy3At9mrdx/w1BxSE0jdW4Uzw/zyePaqlzxqwoLKvaGDLxA8lEYrJ5gyAXXDjbyuQX0usQZJih238NgrfWtLUksvbuTHW3Svv3iyVTkgy65qyDsx1H+WtMaTUQevMEEFF+HlEBFaDl426Y8r91SxNYcdaPXiGVSQSShpAg/XODjKeje152GmE1EPyMwO+c3Yz3o4clp58wYbNx/w7kGqrTqepiyDBEBkYxXXloQe+zzLMarvlw323i6jN+W2wusbJ52ucfI8tUeQQLzKCcnjlycsBNFutMKbZ3j7b8zp9pvR1fMX5t45HhP2UkoRZc/xvCTxUFrHUPYnlryfI+j1jAhordekBGsnAp2wXaE68bvzQR/q2eGFgScKWjfnsjuHpzCngXGSPznBJx35KJ/+tey21b40l712ZsojzicZNOpVASgloe0MV3j01s6QHWXQj1gf/itSwnH5tDfCV3y6/aGu3n6blUU7TxZA+Dm3roRnP08WYG3TqSe26WtKxEE3Eg/rh+z6CWLbuCtJ4PrcToEPFUX4tSuCXpu+a5deyUunrt7Iy+gX368j8VQx6aW8wlKbUa9R27+SaQXzy0wlcu0bdTb27HEZbbmaNivOAxx2Vbi3vTUmiZELiORX44tjeUCrMth/d6wIkRmSHfsrXDlWeRzsbysbSfjqkcFe+Gt07coBgaNjAfWWvIhlbWEGwsuooY/CInGstDlAlVNlI72uGot14IccHC/y3PqhXXRzb/ib9Jw/VV0SqomAaBvq3dAuvRqa29HwYlbFwkb2XbFIVKkgtMMKW7Wxeq2Aq1gZW7WDtUYliWPcmLLfhPUYNvIp+YWMJ71pCDu2BsxPu09sZ4DvRgRNx1av62IvWq/aQR3JbEL3IlloPeeU24WJQ+Qt9eDVfqurHS295B7z829l6KUNKZhcdRsVkjTULyHV7EfOVJnlXTYxZ6rMr3YPc9L7Us/H7r2POfNGsnhv5yv3snoGXNjaynoKmXK20YV0J91ndRyc3ogGV7oIhZB+rCQ2apBj+oPn8B/8Z/K0HnM4pmFj3Vkm+F7ldYQxTuAlAOyZQ1xLb4RQiibffMtyMyJYyCNZlkem+dgz/4PKwKhdRu/hA5EVZt4Lgo2xhPeT2DjImuT/BkW/6fAuQQAA"""
WAVE9_PATCH_SHA256 = "d569f0967b6cb4c11479eaeb22e20ebfb6fcfa1421ab6c35c51a2160401a5df3"
WAVE9_PATCH_GZIP_B64 = """H4sIAAAAAAACCu1b63bbRpL+r6foaI41oAVCJEhRvESJJdub9YmdZCzNTvZotTAINEmMQAACQJGM5XPmIeYJ90m2qroBNC6klMz8mLMb/pBAdHd1d12/qm663mzG2u25lzL7ZO47K9c+SWLn5I7HAfcT+cpKlmenRpRu2PQZnQ68wOUbNnSn7sh1DYP3R3Z3NmTdTmfQ7x+02+1nzXVwfHz8vPlevWLt074+ZMfwd8RevTpgJyfsI79feTFf8iBNxixcpewFG7Lzc9bRmRfAl55J3yaM286Cbduvry+Y49vLKGFemnB/xtKQ2SkRW4ZJygZ9FofrRGfrhedztrHuk5ONlTi2zxNqYcsVdJtydvH+/Y+vL67fvmGrCKgQiThcBe5QC9LwrsU0vkljWwyyY85ibrs6C/gDj9k69tKUBy3joA3D3turwFmM2Tz2XKY53PM13MsJ04bsJQtSzz3pma2WzqgJqUPboN9qsQ0zTwcsXSDtRBB7W+w0XAcJCwMOfdswiAcs8e2pQfMY2/Mu8xIYy1kYe3MvsH3m00KMg2NlUUEYL6EJx5znazuB2fV8NfitNWHvTRbbSQrbS9Y2cHhzsmVJSMSK2QUvbPevtgNSQyn5XsDtmNGCYxdGz8JYyAsmimCuNffmi5SlIBCxtGtYc2T5piWnm/n2nIE0uZPifv0t7MpOmRMCPdhXytnSjiIvmE/gIV3gtnFvdjDnrkGCu1rAqtwx63XOTHbJbCf1HuzUC3HNnsPZMfG53EA6IYa3/3mfA2Y8eIk3BeUzgEHxls19a86XS2u5tK37oUamMBjoZ+x4cAYGgaaAHyOyY3vJjBWocGRt9cpbsITIAoY2vvcC/aBdf42yPThufq83NeQiOWCtA/ZZThXzOfQD/rIX0dfd/jcT2sLZmT5gx2cjac34AU7+xX7grAdaZy+hP4z0kFzCHDsIQnBgvmcLnQ1WyylHmiBGF977oUNyMdRJp7CqF7E131opeATxFEyzJ3Ack1Jv4NyL2MWmTahnj0nxuA0nYtfZQkdjUCQP9HWJ6ttWFC5Ze7/84vMJaLuXLpY89RzYzdL2wCKnXtr2XBCuBypkSD6q6wU2LkOX6/IZpSYf0YIm6gjJVmib5Dz84e3P16Crs9ieo2c8kd4LzQo5l4SzdA3q3o68iKPxueyu7YdhVGfdFGl0Al0+dYMSv2bYZZ041AH+YzNKdmTq3R47Hp3pZieXre8KVSFNeRF3dXZD+ng7aWw3qd0LdjT3qBn18Fayo9Kh4OGNopbYWXRfhg9KT8FhJ7XBL24mTT2Q8XmXreyS8DQyAi664TR6aeaO7PaK2vZMud3XsTQzLI7VbAX6c5LsAiLYHxNYVuyBxwanD7GHnC5D8YKvhWAnfL8BoXNqJ+ASc2pIAAMJF866TREedCMKYRyaINoZxU4YxSgILe070CsMaNucDCh4Gq8ccpFT7uMCgCTfgOckv6xEG5ze521aoBOCr4M+RrGeDvppG+Ktn3oRuMNwhtFZw2UAmZiDPnM2hKiYhESXb8BX4NrUKJyTKzx3O7JdF/tlc+aGmaxmM8/xMC6hsUDstucQIhiw3kcDN4STVOSTeZayHBs6KNZLK0oWfqN3wqfBhOUfwYhzJicAODDoCwrAF2MNbkTOpLitgqI5kTJJhwAVUBvC2Qy0VlAANhiJ8Hk9vfgrCAlT7naHGGi6vQ78y0wZJCDXPsABfZ31usqS84X7dsDlWnOGnMEIBDSkyYIRsWwa6qL9tEYLiYHHAkgRQbSfgqu/q0pitMd+R4ojlbMubdfwQyMRvoh4Rb1oDdkCYNa5H05BU3FyEILAIwREGJlHVZbdri7p9Sb1LQQoyJkXA36UkEZIRJgeEC08eAgAKAZ7AZ1Ox2xqx0ayDRwAj9wV4W+9CGERxAqDRd0uUE4XYC+00oWdHLhPoH5wUQYY9XR3m8T4Z72R0z3rG8Zw2B9yUJenML4cvRPay3YJYwADwF/EACxaTZlwHux76nvF0wxEnJA+EToF12W7APMAls3icMk+fff+9Z/fXFjffXz3xnzzaYIOpxh08ymnNR6j8zNdiwc24Cv30630NuI18DkMfT2L8FmIh9ZwFYEXyAEq+A4DMChwW+BclAdBd3sD7hC9UYF2C2oSkgbJaskJlEttUtDtk9BYgoU8nmVLpl3MLDDosWRd/i7x/JUFvqLWEIcRL16SsfdGOmjScbcPARyBJfOWkV+XBX68meSa+hI/PAL+pH7wlXZ4I4R/y8z2GxlUiE2AWWae77Pv3n74wKQsDluTgs4XsU3aKkxbAP1zBiDY8BIAZUuutdjRESiMOx7z4GE8frBjK0y0Q6kO703r48XV9duPh61ixKSgDDsoCH8u3pc3UX6Pn2Jb7812phu0FUlL7ohppaQnkyp6MEXkrcPyDOoKvxzs5aq7BZDsOe2EcpecqzYmlhR+m1j7453WKE9yiktbL7+JIeupvBJS18urzhlZ6SxVEix+BbnbnKfWDEwYF6cdQloDjRaAyMPWt7VxhdruGpz12EVB6Peu0diajyRX1B2RLzLBBsz9uo91A0Pw4SATVGHnV4tw5RfiIA2R2k9IB4wZ3O62wfy/zeoZJa8BgH1pnZ3KnRAK+1RV8U/oLAgDBQ9eHAYI+6WvQKc6CwoJZc5PO8JttFj7G/IhqgXQ/opU7liaZOFTL+3UWWRa/+k/bygfxA3dgoX+LL8Cdmev2F9uCOjCl/++/gRuj1b57ofrYUEN1DUBtOUAnEuY9sE0uuzaTu7YpS52ftwy2JWNThNsB7YPxpSo/v0KlguuHVPk+6HVAUu3P92y//nb32kuCLHtpf1XmOBP0Miuwgt2n2CY4ZC2IsD0sXayFTrQFzowONX7g/06ALuyXG8J+V/PVFTPC5reIkPkO2Imcf0jTwDZfq0BgP3OfxvHYfxNTQhK3i+lUXFI6Al0trbusWIly1Q6Va3wb/Z9q2fr1eUKdSaENLP9hCvG3CqkXQ2DircTgRiRS4n7YqEQValAk6DIfrr+mVEVoyBH6XqRFGPIWyXcnWSFG66m1A01HCRQUBNLaduA/WX1CIFPHin/cIOVgbXm+F4UbcfjNAytpR1sLTuer6hs2Lot2YnKcd9UuE3monAKOT9mR69RAMVblMSYvf6zyx88h0dpXGoT8tjVvtkzdvPE2O2uhrKeHjfq6fG/vp4CKtynps8Xc0XEtcX+fxTzbki5RwFy0bp8uppbdpLwOLX4/VeaXAuW4nUsxB8qDIdUW0Q4pvSiEn0JpdRoio1QQX8PTZEU9cz296I6K/KjBEmTez+lZPas26dkdo93pxqXnVgQw7VW5XV4Z4UQRcFvao+PGUfG47fBHMC6VloYrAGB4BqcHdbz1Th+aGA1Rmu1Wt8q+0a0qy0RFaBh0EPiiIdN9maTvdm2INpqT9oUbL5dox8KEp74FxClRtNT8eiu0eKfUJ/dlHTUuPEYUzYt17ZWq7J7pER1vAQI3ZRZfyQ5A/GfvcRnK39yrIfQc/Wm/omzp79IQod6F0vRZv/JtCcnG/6qRXi/qnewp/dxrbfMOp4xwa3CawidyskR8HooKyz4pNQ5EozyQ3GIlRdfshOkErmYRz6kOCIdliVILEgWBcgJVdWocvj1OdbyvBQRWFYb7BqFoqLbNURs14q3+Hk508sv6EwK1OyhUDs8nGL5a6F89K7bqg7GBId1G1o6le9HhWYqLa2ydQjGWYJx56xxZRWDoszAEkniecOiK90pgT7fm7xqCkm9tCTcppJeMnRitdHlASVapdGTcpDMpAXyoUXqJd6iz1Y42CryphzI/7SwYTWXMiFqxxxQIWUYY5YflhQl2zRsxp6yatnt6SN23O2edaVJg9eFRCNJk5qb55uIO6l2iLmuwKqHVac0hZVZonwFvD8CUGtdfTg7vTEMHGTRoNvKGGppGFOMMIxbxTUrwU6ZzlhSqpVoh1mJ+bBlOOEqSDVSZ0UIBVLHszNIMcN0kSH1AkWLMr5E2GB0eQVrCdGTByVy6hErFrt4KpA9cUpuLeYplc49PHmFtFeCdKMg9Fs2ZqobeyaBTZUAayCgSGX/Cjp1AuXpURNx59rh0gtkIbmnnCyi8R62fs0+dhyl7hT4/lU9dTbVtLZdtJ4+wPqHqZVPuX4d4/LyeOc36xCWFZGEYfvePOCusRwGw7vuoMT8QascQN/iwRd7Q+cc4Ng8spK80Id2NmxPt5C5SqJ0UkVmKUvhJWp0r0SYE5WHH8ArQbhMUjzmwtC7bIsaMdANQgK4diympT6J0ajxmeMp9pqkhjjWMB5Mg0pgxR6H7BjwM+xz1/FBvAoCyK6rJwf5a3lo0BlNh73O1DB6jmObpr3z0KAYWDsvKJrQp5sjE2szpjhchjdZJofQRDuonrWgr2fJwo540hJeKyvIJU6MnEBZ/fTx7b+9e//eury4fv3v8t5PEtZpwegPHy7+mNAVnvz4EELQUJb6CR5RMWltbxMsxk3xDFIViSw53xkQ5jBaieo1DsTqdHbIaYUzbdiqxuTKlYM0q65gnAUYJV0xKgZhKaW8olPy0UitsToty4gybGTXa7L7OCI0N1PDyzmAmXmEqM5e0BHNTJwNA1uADEQf70FEdJDBglM1E68HNVHDY+IobcNyohjyJXGcfHFyycStnQI0kqXAfrGmadRpEcfrtc9GDlPBm6erOIAxlTqQrF9QqoX5V0PlQqiPh9dLAtX1NFTzqzLt4fUW13MhPo9pR5L/AE3CJDthL1/iaiQGaD5czRf5GT+qfR6n0e/hnSyx5JMt+RQssWE9rYkaGAx4kQee7D61B6sATXe3dKUF1KQN/41G30HY6yRCzdyWvEelQfoPd2jag9OOYQwc2xmcnjb6j+rQkgepNqIP6Q+66EPw31lH9SFS0PJ4UHpKy+X3KxuA5y/cxUoAj3ngcNKdTKBU+FdqpJl2UimzfF54qcAxoY3yKmFxlwEvBXJBr3RFiOFFGwHLwVSdOEzwMs8qLpswllXVCwtgDX+4QS7cHhzXCpuZQYDrgamsfKrCMhDDXuGZmdT9uxam9vNoBV1E3pAZS5YJgK19pbo3xcSU46ur79/9NGaihpVdEKG6SBujmjgLaONZQFYoUW1JzDjJq390IJVtDHaFe3FAz7UjsegjyKEoCxmO4I/Zp3SKqoYqR9SBar2P3cFTXgbQlSJbAjqhF9U18VWW1fBLiYv43oKgAezDRwPSO4uuToKjf8mGk6KnKFex82LDcx/8nzMeS8AwHkudHI/xqGM8FigAvklF1Y7whCRPOIF+VoLpjiAF6xjdHFll1ZZ7zDv/g8MkGCqAC5HtgM3USFTHyVtlzxrLTgR6EiXYUJYRKE7Slg1nsQruEou8utbrl5RHXrTkG1AO18L6kUWXMrUjGnwDXsK8VbXkfk9nEyBJP+/9pdjSBnYieJfLS2UeprFGR+UBIjvcvgbT+TzQlNz4OFtz7f1GvALSffV1bcodjVrOTgBpVSIm3RUO7+Bfwf9SD8AKL8GTmsBfO2GrQb8i0ulqBhu6tB1wK+7lCs9RxuOAr6ULoB23jFWwju1IU3nhrkmLYLxBNzFznsh5ikGGG6VxaWTilEaqnNs7mooOizR0NZfi8tF9fW1qHwzbR4J68x5QA1aRH9qu3O+R5Ak8bco9K7utq8wzNr4pbxwPo+t0hKQbidwZmdHjrWCxYnej0+p0Iq+zXQsDmoVeNDFja8lCU319hXrt3twWE7zfMrgEvLRyhUlnQtBClOo2xWpzBy23mDno/CstQH6RB1zyKm2FAzX49/yFYJ79z1oGaS9mpQDrAnTujXqLSiqFBSnjVzcd4PWk4ghuK/1JOk/2pvndNFyQ+I6yiTJ+N1tbuT+yA5lS64uqMYu5CLi11qyAoARChLQentW1ytDa+MWLtCPfrL4GvdMeNfRarUdmG2mIMT4Bn3IOell8bSkl9UOJ4hC4ydRH/khAZC2yLI4D5a2dDE5QAZMOvSnjFFfxOJ0I4cXTFSxmyyCfBUXB6PenNQ9M47TdMU4vmYa6gfhEUKHLtkvuenjk3R+iC6GGRlQN8XIZIZqfcrwZp+LqWpNE1qNu17Q7I8M47Z3aI7c5M68PLmHrerM4R5G3+Qb6aCTRNd6r1aqniJfh5mt3G8irW/L4jM4VSweLBWb8r+AGbPGWJXdehPcPtAb4CGyVt2J++NHCvCThaSuv5Co3dDKsDhyeB2ECsFctMMuUWrmggMiGpetQUYvEyImh5ZwDsJR5KpfIXBbMKZOgWgEVIPJMFh6XEVgTB4CNleAcQO/AzxmMBKuFuQhiVs4F7qfQ9uhJ/PmIHPeGgNE1TfMw6Pe6EP/PWuwF4OFTCgIeJBVt1jXPxLdhqaKPJGEhq6EKRvHz+ACY+MYb3j4yUGd7hnkAyZFQljjeg+RiDUEnBvt6wFNUcO8ilr/E648pWw119iAifKs0K9qGBpzjfu0iQAvB4o12OAezsFbRoU62IZE3WI+A3RDxDt1wHRxKxE/2c1tN9skL2vGdjFD4qFUzduI48lSJww1d1niWDwD4a2/4DWJBwKI13AwTRNr9FKtsPv5EqXGuTZ2QdMoqlUfvESRNEj3DSl1rP1HXquKydRWYfatG38rYCjJTUD1w5iUzn0OkCpU2v2IBuxASLWDfwO1T2KNxrAIWLYrvZADaETzTKf3uzgQtRTjd9JzVcMKqrLrdS2CjzLbZOxtthjiTz2icqiE8mIqp6mzxqORwzk47DTxLvSVHB+Kb4v7Ho+KyZ4P+Uz67VuxCS7bQZEGPe7uKbXSIuavxCRzW9BFJ//4+QrRP9gH+PtVn8ww6m2fQ2T7Ro4Im93cuQ839fes4tOlT08X9x8c7xfe77P6VZNdUnq69qucfTfTQfSQpRHuOKcg7CPBUnwrCtdZUBlc9g3BJv3uH373D77L7P+odfrzTpHMwuG9HCR6+YU6QcCexAFlQLbTLBwCwhTeAZcPrSjb/pQGyYKZmrRDRIHjR6CZ7bQny50NFN7xIXOu1/9c+oKVt32xnl60oP/lyKzK/z/j3i6i/fJYrGhvdL5CGscf8l2OfaQnFe5f7qc0+j48N88uLwwYRQU5udLDeK9bezjbbAjbJx90/G8rKK3h8l1ppqGGCU/5dUfUnHez8H/jkREzXYIJddHnrll04Dvd5jL+6ZVcp9307Xi3zi154BoK5MSTDU1787hc/1+LEXR4W0281EvUIOv9tu/gF4JuPFx/EDwLxFIwmTw7+F3gXTM33RAAA"""
WAVE10_PATCH_SHA256 = "37d7df45dd48eb02cf10d8e3a94e8f210d679eaea4e4134d8eb63a8bb5e45ce3"
WAVE10_PATCH_GZIP_B64 = """H4sIAAAAAAACCu0923bbOJLv/gq058RNxRQtUbKsS5xu57K9vZ1bJ+nLHq+XoURI5pgiZZKypE5yzn7EfuF+yVYVQBK8SXI6Z2fO7OjBkgmgABQKdQfouNMpazZnbszsk5k3WTr2CV/b84XHo5Mx9yfXRhixcW3Rges7fM0G7bZptwaG4Qwm4073jLVbrV63e9BsNrfAPTg+Pt4G+/vvWbPX6+k9doxfgwGDJ1OfzW3X1xqs+Zi95dHSix9pDZ09CdaPnI3PotgZDnkYBuFw+By/Hj9mHw9Y8lmErh97/jfa4X/4l/O5fcWiG3ex4A7THH7nTjgbcy9YsWhunZ2yIGQ/vHj6y7ML69Vr6+XLCxbxuHHYGAl4nw/YwTH+ODlhv9l3nA2Y49ozP4hidzJkrsN9+GF7bBa6Tq/L7NCNr+ccnrEV/GLxKmBP31+w0I5iHkZGCsyPg5tzs9uF0QbOcsIjNg2WIYs8exzpMP14cu36MxZfc1kjdgMff84XMZvxAPoINxKcO2U3xrUdWTBZQNpH8RQ/Ho+pJ3bOoK9l5P7BR/nS2zGUfXKHjAo/IcbdPvvINE1z2UPW7rTZMTtrsAfMPD1tMDtibsdkTdY2z8R/ffa5ABIGsuyz8+whfj7dDdnRpdu/+sSWfmRPOfRB6xh5sCLD4RRmZoX2ylrYYRxpdwZAWcShRp08nAR+FLNlX2d3hseBMhq5XqewiBpgjns6C5ax5bhznbk+fjfgm11qhzM75tZycaizbr8nMKGz/qBHv4C2tEMnWPmH9EzUaVypmEwmN7dDxOZ4OTXwp9YYlev4iFPRPTthHbOiyuo2GrJf+eSR238MlbWWYciBA9LlyKGDhXY7bhiTwPP4JK7sa10GRAueh/LJ/QQrTSt6BuvZaWwH6lgwPjlL2/OCiQb/C8TjciwBOd8ZDixPddtokmurZTMDzDxk5j5A1oUBrO8xgHV+ANa0Y2oSKf54W8NNXTM5g+q2yNqM6zhwNMKbLjaAdgS/G43vtlWOJjo7uuOTby5b685k2R+xIqqutgJYK72tt/ZGkyHMpD0ap/BsxFLEiK7KaHGRcQFiTlsVOIvdOUcG4plDNg4C75PCsqe97i6enYOX7GQLtyzQcaeqguR3nllXiJ8bY8bnc2SH1m3f8kytvip+jhBN+vY6Yml31gH87qqz3gPOeg84mx01ElLC/dIxd1SW3GqvukQxO2uWaDH5fGbci/jey/fPtft7Wrvy44pHxHWiDah4YeCDeNUq4SH7iGKQ9qCYnbMfQcDbfjwc+sGqJJOKnEGwpH9yh39yh3+u3T8od3h9o0nmYHDPXkTc0RpoE0R8ElmgWYAiCNYJ74GCLbgBDBseN/KQPleoLGipWUvUaFB50aY20FtpCFjRM5Vqcbgs10otzfL4Dy+RSpue2RSmH/tI9snnK2H5fcS/n2ks7KMc0dBofwYzjH2Cx8ES7dWPNITsucO92GYfh8eG+fnBYcUSgU1utAAxmhh7M5lsA9Akf+YbFTktar8hBxPYigMNDRy1glxM+BLGMZix53/ikwIxHYMJdIV8GfErdjGZcI+HNti87F3MPc8Ol3P2Bkxczp6wCXc9tI3BGB5zI4WCn/dgLqMN/8Pzly9BuITcnkeM25NrtuLu7Dpm09CezcFmZ2hvsmdvL16ywJ9wZvsOo86FS6Jt9gfok2h3+qbe7XwFp0StQ6GfdyhEgQeWKvPsJWyQZmLoswmUk0cB9j6bXNv+THoHUmA/960Wiya2x6GyH4f2JDZgcPE1s0NAwu0SpCvsNvIoRDZo7Wa3uwZjl0FF9w5R7Y9SYFhpAjhxHRwMWJCARcIf+oIW3Ee3B/upY7IxGEo3EeAwDgSew2CFDg/pmSi4IjQoBYsFTE+gSDBWE5+EYokXfAlABrhcuA2xLZA2Ni5U2maTk41cMusSsGWDLm8xrTNryRTmUtIyb5iRJyVvsKYjr7NXSf/B9Ypqh0feg+qWiVVmtlo5q+3GSFYapaOQfwymgZai6E1nuaFhB8X513JnMvm2aWs7tbQ/O77P9xnqyg4XkoffS4rUY9QCKpzEds3AiUSl/CRC/Ztj+CuM914YF919Ic53iNXDy9t+U3RwlXCvIa1xU8hRudyZvITKTfS+fkyHlZRpKETbnx80ilI0laDZTJoJHaEUlT+zRura1ErPry4yOwZ7J9chgCE+i4NnzAaW/GtzFQJWQdjb/tJDb/SGadfcdlAJPO91GbAX5BqnPfakoauyEyTBJFi4wI0WoK2AdsJ99j//9d8gkYQTmodNKN8wFE/+ZCPkSAw867sDpyLCEIWTkxse+qDNy0fGIl5noYDq8iTWcNbt9wddwzAn5sCempWxhhoIuYBDTR0U8Z3eKUp4/Br0UcD//MvFq/fWs9evng8FXkIeg9AmgQ0oan69D8GbeVZ5nw7ZeBPzZhZWIKku65GMFmGFwOcoZU+glUHQnqPsDaa0KkJSI6GCYF75sMH8arlNWo/t3Nk+xh/GG9YfETAEogQysFZ0vZxOQbUIQgeoA7YnDjDwYfk3IlhRmI/x9XFm3LmRO4YxGNA3qESVCJQsw1jYoT1nBghetrDWetXT1FjLP5a8sVAEKFtYpL1UFSDrPDgG20OyYmACM6gSggr/YPHo9PFIfTyGJg/CR+1e/jFuzAfTR+1WoTYM6kHoPGp3H6dSyXNE9zRiKGzr7BJmeTWqLjap+DaqK+9QuZh3uQ6NVvSA86+pIPpAPFylw5wHd7IUengAi+M6xnpUsFiASgCsaACMcwHGvWi0gD5Fy7At+/weHrJxaMOuePuUduqo2FMX6sfUjygB2pUIP9VFcaddHAKOwbN9qUpH16GE1ZMtTksNsAXuL5TBZ8VmZ7rAR3UzRWlGPpvOfb70DC+QIPrp1M1RGVmJfsTGYA8VETDAVr1RlXkOrcW2RxYCCgT1LdpP7mLbiANj5gVj20sIA5EJ1DXaVofQ6phb6xAinQ4RBi3dkxevn/5kvXj9+s2wauVNPZnIWbbyZvXK206GtnZLNuwQhNOEBhzHiJQK4m9/lCF+BQxXwnDOkipdtbmYCK2LRMtZthHkbMUOxq2CVbN9YI8jWYbDmib4jK6nHilTBkYeBZVOcdGxWruHlKqz1noqP+mE1yow+tPZB2L/awPsfm2A5tcG2P5TAF1nLeF1Za1WBTzRynHvjNCXQHFLYJu2eWa0FA4lSmFhW9OW/IwU8ue3osaChk9Aehn5dzLyf/f+9duE/sPJIuu4Rw2zbUYVhxmZpxXPdEGoWQ+wcaHYxX0i6oRtKk+ofO76yR4ivox/YX4KQoul6ENRgEf9pIYpaowU3oOMItl/A7H/yptPbFvJcAYJ5uKU1/Rp37VbVwTATOETcv2Et9DmBRitDLXdDLWvnv/+vpKzdFKOTPxhUMM85PQ7ldxDzFxyw3a7NAPEO03BvCosJA5rWGRmA8nsEiynk8i4awZBqLXHqVp7TBZJhUo14/M7VKdamlCC88rRSq96uiZdaj8rwIrmZ6e7TIG0krQH+s7YGcDEDd4d2O1pf097IAOzwyjIKqJlcNrV++wY/mIy0gGKzbf8dumGwikxRPc7e8D6YMghRwBB+oChUQX/jYQPbENG58Sz56CBu3HEPVDMA7DPCNgcXXmAN+ELW127gH/E4Mla6mDCLJ8vodqYs4sXsJwX758/Y8sFQDkQSsDSd/qUnNBgGl/HsPTClg85rK/t6Mznd6BeoDEIqnrDOGhCsxfkVBwKr7OGblRMzQCjVutjBgDoTSdg+zd08rCK1IcTGGmjwdZkMcbXCDsSwJ5nMyUjA02SXrcpDEdMYzKoH2Nz3k5syCB0Zy7YDNK7KWyEZFB+EM5lHtV5OrYT6F1PR4P/NUbshSnTqVi0Qs/k+mTDokBYLmnvAhe281d7gqoSrJLn+twOKRtLWDHoSaH1go4W0Jd0EcewIGJo6E1eWJ5pye6mnj0DdoJpMzhfD41gMLonAcCDeQlP6cL1ZyO0ia5x2jg3259xx6CFe3cNo3KGrNM6M9kTxfXKKA2KHQvLPFdANCGaf017qmbzZ4E1kZqnn7Hj3pnep61Q3vqbIkMgewgQWvnc9fWDZvkxrm2VWYXPK+2tdEkOWOMg8bLnjS6yl3AKZ2do5p8N5G5WXe8doDp7ztHzP3MpPw+d3n4ADMxzbUGz/nI+5gjzmoxldNHSuhhqp8LCsGYbKyYhhb/8cfILGMcoV1uIAyxaB3ryM8p+boJRIelwCITkAr3OkXybCsFFK/ePPzw+Uu116M92YUeO3TjzJBhlExTROA8crsvfuGryJ+6gUYUtC2WjFIcolYBWk5DKieReuK3IORRMYzCSeHPhLjhuPofdNL0gWJRRN0YYLV+Xv9p+Dl+kj6yiCVWAbyzGlR2YervDjgdnutlK17bScIWZXY0qy4Xd6vo1xcI0RjqssXszHF4qZFllA2cYTizhqhqI+LTKZlSlwUA1PdezosrAwy1dbrZVzPUMg2OlvQL1RfjoGiTYtxEMK3SBYwPTR6MU2zNcXvQOBZL3GyA6wUoFlphCQwAoSLhg1k2S8EAbiwC9SugOhn1GshMjXCSE5vYN0BUKtE0KBgg8DmVWrEznBZB8DZzT2+SlDXbv8SYNMA2NZeNpIZ+2UZWL3QW6uKYonTUcBoAJOdAzZ32QilEg3Gtr4BU4NlUKp+Ayzt1cgK6G9ZI+040ZLadTd+JSGBI2C8huewYiggHqPRE8axYoI+Es+XWsqKDsXibMGa+SO+Ev1UsgEHHOZAegDvS6AkJRxc3YVgYRvRS0JnEfVAWkhmA6BaoVEBQVuKNnfwUgsZXb7T4KmnanBV/JVs58N73dvhuWRwhq6b50BLG8f6Yvtfh6t45wzZCvpLgSgy37d6AwUtmrNCKivHtCjiEZADmD0QIQPqVzqY+QIkKO23VxLaWZgfZohT/Nx4WcuiFGjIVKI1ZEbD0AmnFwDNqGsF+ApuMhG9shmb6gPHJHiL/VdQCDIFQYbNFuY1jnGvYLjfTajnZq/cCicscLymVSxz/rDCbts65hnPZOOxNnvFPHl61rVXtZLtUYGPoxftEBg8VyzAT7YD9R7Xc8TtSIE6Io0k+BedmOyA+gdIEP8qDAD29/fGY++zBClpM1uvyQwhoOkf2ZjsV9GzQs58OV5DfisUiV1RMZDxz2miOrgQWIJ9fIC4HruJ7HMB+BFGvQCptC+yb9FclTURFFFCABRoo5rQ9YesRvclwLHZC5gELCnkbM4RMQKBjV2UQZPKiSY6cwniaBT7V4rKmGMAqTE8cmkpyWVP+GwRigYgMxCTUeOyHLxF7DrJHZZsp8Bk1q3H60nHNCjdwsivK+U/OXQ07FdTJkEV+1gF8NJV2kzyLXW1rACksFYbDgpYdoRJceKsGKrOy4VJaisQImGeb1JaBDVrabV5QS1+0M9DZmuXRBhaKN4c4XXnlLyKxKQbzqQ/zw7LjNpdiFV8xsPpPSnRY0IWfKypFbIj1hUwgxUyZFggTgNiKtxr8bDu/s0Aoi7VBuwp/71tvXv8GaHjYMN4LJzXkuTg4DzuAUQuRVY6ZIWxIwTkacrEuYDJtphQAdbqdkB9GxoeOaSWVm5DkDEysbMzs6qp/lC9N6e/Hu/fO39bPMANfOsjI9Tc77hdlMtiatj4SVzjdnUiebChmQsuMah/VZZZ8PtqLd2YAJ5k6aEVnGKeLtWEYYK+nl9Y1WSaQkcue2nn8SAvcsPBKkXAj9p+RSeJ4iuABEcgqQM0uPGzMeW1NgiDho7RCMaSjEZJ7Dxneldhk3qWuc1KiDINhOXWssrWspeFNdS+IlNS1zDKwOgFKJYBzXwkgZ3R6gZN26+Ui+uHVSWGVre8Efd8LAajVw5vvBmefhkILSpnTDXrelt83tjBi9iYag31JGITqYgqWXbSPa2ZIVk/0DzI0i/WWp+V3i5cwJWy5PSYrpkG32ociaPqCMJR3Dv3PDwEdngBSxqGhN/WwHJQqRdoTToFRKFL0q56L5ZQ6eYvrMCXsilSTiVh/+/ZK8RDihK+Csv8t/waJn37PfLsn8hX/+8/2HRJH58dX7fgYN2EwENtgEjLyIaS9No83e29ENe6KLmR83DPYO0yVhS4UwfUxXUnW+dzBcUPfUVf1wRYkz2BfKk7n9V+iAxMu74ALTBEViKpmdHup0G0EDXUEDZz2929tOAzIBfCgStQup3sWniJChktJdTGD9wRPpqqVFUE9JiNXQyqfLdLZC7z/8TZLKKBrA1un/m9KZTJ2JRaLUa4VFNLLVLmqPipQSaifaMznsi4GCpk1uW8pwffP+d0a+zQwcOfGU1BbQFEHJdkaJO5erjrYKzy4CyKDJLF0bdGvpU0ZzKFUw/3KJ/sKVNvHcxWIzHMZBYM1tf2PZ4Wwp8k+vcvuk9lwKbRc9n4g3ZEdP86cbcCWG7Okv4pTzIg5zZWI96srXW9qud7Td1BXk6fS4kk6P//7pFHP/t5Dp/stcWOLSYP8/LnO9JbaFALLDMny8nFl2FPEwtvjtN+lZ4weY8NDS2aGCcDCIhYRjSi0K3OW0yxJMeYTmAeWV1MIUrpKO2fxJpuGLNB8ETez9lFxcZ+Ypubi2cHfyfNuRBTJcaxQeBzdWAFIU+Kb26VOCkeHwOZ4H4FpuYDAGVOCTQwOqHD800EerNejAcO7MizZf0uF0ndEPPC6MP9bJk3XyZEMJ/Dv3FEy+WYIfCBCu+PIJUuXWKybyV7UWX4J86iHpSHHilgEtpbZGozB7hETefUxfvsyj/khihu4jwN9W+mti3QWuo1fVjyZb6gvXVJ9s8LPOqd7u7yYMAhvcaxDuvWr7W2ofl2pLa3GPDq4UXOOZnyyeDLjuS78r/lK8nxFK+b4Ibacu2SSunAMX8oVnYxYrqawiMIFhiiwsMSJfO8UTHp2jh9+NUQNLIgZtIyNUyhMWsl3LnuLn4VTPP6BINZDZXUZ2GLJm6WNBfPSs3Sg2RsMUs5RKJa3C/0cZZSoljfzuEIizBOLOWeXIiicN0DKwhHF/XjHoQnXy5pxvdTpoCkg9NyScpuIWqDznqeUb5GDlWhfPU8jVgvWhQeo53CLPVjDYyOymVJFPzpYJg0icQyMLQ73jJQ3kxEG17kk7ug+MHjb0wDzTgcq2bmh16KTK5AxkXaEuH5MaG+mEtN8u3r7ZPb1UUf05Of1lKybJ3I5Dd50llFc4mdGznDmUM3hKOjlFD/hkGcvNJ5zOIHjQM2Z7s4C0bYqeyZ2NHrQor0hLgenGIm0jzZqQ6TFS5Z8ESwyZgVyLhEMeGcND7AcWBH/nrc7a7PD9Va11ncpTr0Vt16FwlEXtB7Om76v4qjrKN3Ssjj0m1aQ8aRnNsWPYwrBlaamhJOerLKk8BQUajwaR+pN/3ir8X9V7lsyE2wdIQypKWSL0YfVhmlTgr4WIT1QQgWAS9bnjTJVtsXdLtMNJWNRMOYVYPEmYVwDK8m59L+m4VVsoV5cJFPeSvzi/e7UgNOzT4qqGx+ZhltlWtRNVIl0Ku3wRJSxWl7WqppAIwq2OgzfSBea40YIOqEwTL4wSOVMDYcJxkJ16ddUY2CJuAuWi98DhnjvGs8mcON/EWzoyZe+8jUxOBtEky0oP8ObY3Sy0F8DO7EW8pIQ9TAsAphcHS/Rs1XMxwbH/cbgYKBJEPxm3ODpikpm1i7qBqFnm6bUnG+UW36Vz0I1d8nyxAfif3HAHXe8atS5YXBX3UpSMMGWMiYzFo80suOPh1MPZAZZSE6xwYUHpjoHitKvn6ze2HdM/YR/waqs1e8hW6AmlrdCFrSDUnYh2p9AKGgb7sPqAmvGHy4IldfXBUACKZx/SPNV8Bg0wd416ECdO6Gi3bL2Xu4TCky2TcvfabcrIhUdgvTLQHeKoaBWVQiZ6uTwNilSUpWGPokArRDS2FydRiroOKASxrZBiAtUV5moFgZ0+HWBsm+2ubvZq0aMIdTuOfTrEZYlom0WamEaW8cuL32GTvgLloHRTwV8uEWjmwgIIFh6NTYOj1gIPvYZ3PLJAA7RuOqZFx2QTfbl8J+E4cJAg37z/PY9QI1p4bmxhyoV2uN+Bv8PCDjL4eoH32h0msVw1hgtgSvXb9xtCliBf27PP13FNZ+odasrKID4MOkcJ63FYPtRyiPf1gYBARnrauB8MeY5FBdEug5DtpY4GAOqPtBw29mhecTBlr3aFgyh7tak4E1Hd7ptCQ0Psg/0qJ9lR+9WG6dv52jl+TCElkYK0DDHtHTWONHOWXHUiMtXEyFQSfqPz2DH7tx/fY1pQBm1qh2zKVxxtOrw3FCUCqCKT5BZRG7uyN2R5McxpRLlEyZZzgJ7yZbHNRT5eu6MPkLn0TOmWquW92YZDn44k+4JjDU8qWiIxC7b9Eex7693Ls9NLw8BGFjW6KrShkoo2WQvDuFLci+pWyLrLdkSSPFm7EbJoE2aFR8CkwDBOTc8kEiQSVGWUCFTANHlpDoYN93Pg1MMDmOfEY6FkEqbk1JIkLbR9A89JAk1G9R7fd2JmHZPYAmBdBFAlQ5RV2T6CVhlAvvtsr2SHyzpKzjw6oKr2Wv08ag4J7OZ8laPalXVdw5oqYe1Ozf7T0PL52/dDXJr42fpiGsKUJhI3tufOfDBi5n2/f9Pu5ZDfa+SdwM8xpZs9owxesOpc2iVpkhHus36TeJYESlyNtqVM8sxBI01UbCdyctwBVwKOGsXIQdGbNW+K9ECA6wekOgPfpG6pTmRUUnzCeBTZmh7Zu5NCKptjH6/XRczVJcaGS9/nYSknNn0s02Fbg3G/0xobhtnpTSa9QW06bNawlAmbFSFPNwcm6oumODYhbrQixRI1Qu2gmEWMvJ6BfFxwMILEUShpUUeTkAxqWKs3b5//y48vXlhPLt4//VdpdEVBGZa4luvbiA6npYnxcQDoEvllZPRRQsTK3kSYUDLG7Hp1SSpuupbWImbGJcaHFUy1fqPKTlMO06R3YMlbuyUrRsKgeICSIqCTf7QSWmVmnEyFkWIjOTiWnDQT9lY1NDx2xn7ifIEi3r6m1OOpOPUAaAEwIH3cOy7vNkNHAaoEePCtChpm7AqnhXJ9+MXJE3F5GM8CH7RTYL6Yl2OUYRHGy/k7jbobGmH/LUO/fMemjMFTuBBjiBXRd0E+Lh6c8qtuAP18wOrWtJP5bYY0I4l/UE2CKDk7kj+eWAksvg6D5ew6Pb2CZJ/KaeR7eNpQDPlkQzwFLQz07VRBS82i+vMoeFkdGUN4WAvIpAnfhji3NmhTvHjQV+LFPyyWL0H6eEUdLLneQaZGJSkyeASUrnK7Xvo33wovVYPR6b/iZ4EBkG+02Lq9udPZu/fWzz/9qld1I66YCueRhSc2BfMQi7uYWmsd+t4AzyFTEyvQ3WWizNeZchMlQxB8gb4L5dapZnV3Zf+HBAhfSEfwlRCS5rOHjC5+r7jLqv6eLOla2wHcT0efzSQXQ88lmabMVUC90dmRwM7qloI2t+rNnPWd4pM8lu7b3Q11d3O3d383f66/u/v2d5fvT4TROkj//V5Lb3f33gB0EgzNJJ/CTUD/grDZKgS9mofpKS4XRSNDjSViTuB/G1fCc4IlsLsmSXeDrQKmBU3gp39tCC4MAivJSqNDb7Hl+MaWvcW9mPbW8xfvcW/dh9hxQ9WS+631xQRff83rjqtzkzFtrbH1QtxsFvV1/N3X0d7uvt+2koY/N0ZblsrxaaUuXtcywS30HxD9F4i/bgXhJ5JUxQ7okQTon53uKQHq6KyGh4MWluPc6UCqWOgXyoHp9P+1GNhOY7MlrdIPv3wJjVnyjSZEateu43B/L1aLzapobUC01u/sy223T20/Ckxc9hm+xejgB06M1lBMrfGFxJHCqyKPMuj/G75Jg/qb8808zXxtxvns1RcRNTq8q5hn3TpWMs8q05tclycLuhEzZ3wXCqT57fRNu3faMgynb5stp/rGmWLTnAFeLCS36qBLIRv4Oj3LTHAZ/JH+BcvhCb05mAPKQ46xiEaCTumamHhg2WhHswA22tEK6uvs+Zt3eCwOVJNfRZJqcuolu8YyvXQ6MQMp711e4sLD6NpdDKW1RGc8yafC54BJ9Wg92s4CWnKYNEvqoRtG1XyeQlpQenxUepDQxhKw+oNeE0+agy0X4YFVHk5ctLjMfuHGPDQSMe6evCAL35wlYOIJejBi04hVZRaQ5UYU+bLSzC4rDtLIlRhWZuCiK/odHrtLqLWBOSSzxRKqiHByYvMmSWlVN113Svdc053dK+6bxmmzZZw+SfYjYCDxNyQZKXh/NZ49udOUO7DxHrK+TfdwGa1GdmHIGzD5uTMTWIzEeVm6qQCwdRLyxBUQcrooNgJY8zkRwdyN8I6T1AuwvuyYhtHrXhno9dFaWS/rS3gKY+oYp9mTU3zSFI+y4d+KGGPl9d3pBdiWSOw6T2srL82i6IEEoa3lK6Aesi47Tms/ZCb8p0LCcvG03cPLhltmN3lv1CiP2vFyCoCf2JMb7jtPlnh4Yzj0+Sq5MJl6aBhLH+2H9OQlvS0KV2W58ALbSSofSYjwa61WvbXQsa9eDp4MXd4NnsJXb/om9BRb0vXgylTrm94itX9hp4WWuzsVjSvv9nYwUwGnoYvZ6EwdBzDsInK33V8tgBELiMRX7RXWmUArdlC+xVqtkW4kJw6ucebiepDQXonwGV6CbLCnwRxZu2RAwOjYgiJqfqR4fQB5KTR6ux9dkCEmv0CSk2YjuQZjPl8EoQ2MF/tcBaGjsrlX9qskuzEhXUKqhRVxb8hb6lt0SX22j7pXo2IrDN/v24pQlSBCOyp0Kxe2UYngYqu0W7mClZtKAEe5IV5qBwv5mPhC2ifdea7Jt9qtP7G1AbwbG2gV77RL90EVyHRA9wKpBAYyyLoycJC8ucwDud4yofWwkWOPWdaf3Oi5BVG2XHEZZSOxUXc1qliPrFO5LfdZxKxTuf0q1zADXcZ6Vnbvdcw6rwWL2cpfuJbFzDdlaQvzUTTldKEVdSdZZ5kEl9zNCKx0GnIu+FgObZQWcCJvm8MDopSjiDoc0zDk586WeMP7bYQyjmOEEzMFQK4lebCkook7uFPdjAAqeiRL9chEH7sI3iWebXSXUTQEL63T5RFGjGWAUh4s8aokNTWg22ujDnvaNvWzlhpGkr5+efPJnsrs56JmOqjQTNPbQp4oEXn5RlUx/eyiJtJqBLzc/WfEVUUCOjDmSRhE8o2ruSgOng5Vb2Mq6JL5mEYSE0F10o0zbfLP6I7ulH1T8y5X5faEdz/9+GbIKl5m20S/ZzlxJJfrLXrMklJSkwwnBrPCuaDumGo1YJ/RYQp6M6rZ7SZUm8OI2lDNQmU38Cs9/KArZwWFQpweEhT/ytOBpCKrWMTn1sJGFok/Dce9s+heyD5qg31VWRSmgvL62ZkHJuJkOJQx4+FQ0uRwSNcGDEUgGP6ThKodCWW7+EJW0LQHYJm2jHajUZSqKEyBIQ2HGC0ELIB4B7uvBKJRycb3awtymQLox8lbQERqJYZKacoGRXkiiww1rdPNEY+8RZKvgTgci162SzdOakfUGKSHYV6pVHK7pTKYBZ3uVSNPQsmLdgTu0vVSkZe3VXKaffJ212wAx8mYS88VK0B9XOqyplBT3sxbAmLSRajqO1+LNXrdL7QodhoUxXfuFt94W6uwO8U37qqY29paeZkshWaPbqNqRUDWodfHSsGzyyjabhM5xRf8lklmj4nXvek3B0esdCWQKq8h2hgOHXh16LRr3cC22hg4uo0lz8ttfaVw7eQ2mOPzJY3rXpAop0cLLZZSnaYYrV58uWDh/YEVrwiUWCibcbW3GewaCKZafa1hbDf21I0rF0so0+qrkeVQiio4rc7O2hVqtMCzwPc+ajeiA5FSqptqlSWlUs0hUwQhZjUIBbmQlvyHu9COPLP4GOhO+6Qh12p8YnamSuOZ+bGiWas6sdTispfeJyqxCJkmZwChoaIj55RgkXQk7hnkdKYCj+ws6V0z4dIHQqHzC6rvTEPaQP3kQB5PhH7n3HFRV6Z3udP9yScH/wvOZam/DIEAAA=="""

NOTEBOOK_BUILD = "wave10-rowcta-raster-factorial-v2"
HF_REPO = "Qwen/Qwen2.5-0.5B-Instruct-GGUF"
HF_REVISION = "9217f5db79a29953eb74d5343926648285ec7e67"
HF_FILENAME = "qwen2.5-0.5b-instruct-q4_k_m.gguf"
HF_EXPECTED_BYTES = 491400032
HF_EXPECTED_SHA256 = "74a4da8c9fdbcd15bd1f6d01d621410d31c6fc00986f5eb687824e7b93d7a9db"
MODEL_PATH = ""

TARGET_PREFILL_TPS = 15_000.0
PRODUCTION_REPEATS = 4
COLD_ITERS = 3
WARMUP_ITERS = 3
MEASURE_ITERS = 10
RUN_NCU = True

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ROOT = WORK / f"glcuda-ceiling-wave10-{RUN_ID}"
META_REPO = ROOT / "meta"
RESULTS = ROOT / "results"
ARM_DIRS = {name: ROOT / name for name in ("wave4", "wave8", "wave9", "wave10")}
ARM_TARGETS = {name: ROOT / f"target-{name}" for name in ARM_DIRS}
for path in (ROOT, RESULTS):
    path.mkdir(parents=True, exist_ok=False)

def run(cmd, cwd=None, env=None, timeout=7200, check=True):
    merged = dict(os.environ)
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    proc = subprocess.run(
        [str(x) for x in cmd], cwd=str(cwd) if cwd else None, env=merged,
        capture_output=True, text=True, errors="replace", timeout=timeout,
        stdin=subprocess.DEVNULL,
    )
    if check and proc.returncode:
        tail = (proc.stdout + "\n" + proc.stderr)[-5000:]
        raise RuntimeError(f"command failed ({proc.returncode}): {' '.join(map(str, cmd))}\n{tail}")
    return proc

def save_log(name, proc):
    path = RESULTS / name
    path.write_text(
        f"$ {' '.join(map(str, proc.args))}\nexit={proc.returncode}\n\n"
        f"--- stdout ---\n{proc.stdout}\n--- stderr ---\n{proc.stderr}",
        encoding="utf-8",
    )
    return path

gpu_proc = run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total,driver_version", "--format=csv,noheader,nounits"], timeout=60)
gpu_rows = [line.strip() for line in gpu_proc.stdout.splitlines() if line.strip()]
if not gpu_rows:
    raise SystemExit("No NVIDIA GPU is visible. Enable a Kaggle GPU accelerator.")
print("Visible GPUs:")
for row in gpu_rows:
    print(" ", row)
first = [x.strip() for x in gpu_rows[0].split(",")]
if len(first) < 5 or "T4" not in first[1] or first[2] != "7.5":
    raise SystemExit(f"GPU 0 must be NVIDIA T4 compute capability 7.5; got: {gpu_rows[0]}")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

if shutil.which("cargo") is None:
    installer = run(["bash", "-lc", "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal"], timeout=1200)
    save_log("rustup-install.log", installer)
os.environ["PATH"] = str(Path.home() / ".cargo" / "bin") + os.pathsep + os.environ["PATH"]
if shutil.which("cargo") is None:
    raise SystemExit("Rust installation did not expose cargo.")

clone = run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, META_REPO], timeout=1800)
save_log("git-clone.log", clone)
have_rev = run(["git", "cat-file", "-e", f"{BASE_REV}^{{commit}}"], cwd=META_REPO, check=False)
if have_rev.returncode:
    fetch = run(["git", "fetch", "--depth", "1", "origin", BASE_REV], cwd=META_REPO, timeout=1800)
    save_log("git-fetch-base.log", fetch)
for tree in ARM_DIRS.values():
    run(["git", "worktree", "add", "--detach", tree, BASE_REV], cwd=META_REPO)
if any(run(["git", "rev-parse", "HEAD"], cwd=tree).stdout.strip() != BASE_REV for tree in ARM_DIRS.values()):
    raise SystemExit("One or more arm checkouts do not match BASE_REV")

def decode_patch(payload, expected_sha, name):
    raw = gzip.decompress(base64.b64decode(payload, validate=True))
    digest = hashlib.sha256(raw).hexdigest()
    if digest != expected_sha:
        raise SystemExit(f"Embedded {name} patch digest mismatch: {digest}")
    path = RESULTS / f"{name}.patch"
    path.write_bytes(raw)
    return path

patch_files = {
    "wave3": decode_patch(WAVE3_PATCH_GZIP_B64, WAVE3_PATCH_SHA256, "wave3"),
    "wave4": decode_patch(WAVE4_PATCH_GZIP_B64, WAVE4_PATCH_SHA256, "wave4"),
    "wave8": decode_patch(WAVE8_PATCH_GZIP_B64, WAVE8_PATCH_SHA256, "wave8"),
    "wave9": decode_patch(WAVE9_PATCH_GZIP_B64, WAVE9_PATCH_SHA256, "wave9"),
    "wave10": decode_patch(WAVE10_PATCH_GZIP_B64, WAVE10_PATCH_SHA256, "wave10"),
}
for tree in ARM_DIRS.values():
    for name in ("wave3", "wave4"):
        run(["git", "apply", "--check", patch_files[name]], cwd=tree)
        run(["git", "apply", "--whitespace=nowarn", patch_files[name]], cwd=tree)
for arm in ("wave8", "wave9", "wave10"):
    run(["git", "apply", "--check", patch_files[arm]], cwd=ARM_DIRS[arm])
    run(["git", "apply", "--whitespace=nowarn", patch_files[arm]], cwd=ARM_DIRS[arm])
    run(["git", "apply", "--check", "--reverse", patch_files[arm]], cwd=ARM_DIRS[arm])

MARKERS = {}
for arm, tree in ARM_DIRS.items():
    main = (tree / "glcuda/src/kernels/glcuda.ptx").read_text(encoding="ascii")
    sm75 = (tree / "glcuda/src/kernels/glcuda_sm75.ptx").read_text(encoding="ascii")
    mod = (tree / "glcuda/src/kernels/mod.rs").read_text(encoding="utf-8")
    runner = (tree / "glcuda/src/runner.rs").read_text(encoding="utf-8")
    parity = (tree / "glcuda/tests/parity.rs").read_text(encoding="utf-8")
    rowcta = 'var_os("GLCUDA_Q8_ROWCTA")' in mod
    raster = 'var_os("GLCUDA_L2_RASTER")' in mod
    expected_rowcta = arm in ("wave8", "wave10")
    expected_raster = arm in ("wave9", "wave10")
    MARKERS[arm] = {
        "rowcta": rowcta, "raster": raster,
        "rowcta_entry": ".visible .entry gl_quantize_q8_rowcta(" in main,
        "raster_param": ".param .u32 p_l2_raster" in sm75,
        "rowcta_dispatch": "if self.q8_rowcta && rows > 1" in mod,
        "raster_dispatch": "l2_raster_enabled()" in runner,
        "rowcta_test": "quantize_q8_rowcta_is_byte_identical_to_the_k32_kernel" in parity,
        "raster_test": "gemm_mma_q8_l2_raster_is_bit_identical" in parity,
        "rejected_wave567": any(token in (main + sm75 + mod + runner).lower() for token in ("w8pc", "glcuda_r128", "gl_gemm_mma_q8_r128")),
        "cp_async": any(line.lstrip().startswith("cp.async") for text in (main, sm75) for line in text.splitlines()),
    }
    if rowcta != expected_rowcta or raster != expected_raster:
        raise SystemExit(f"{arm} factorial marker mismatch: {MARKERS[arm]}")
    for key in ("rowcta_entry", "rowcta_dispatch", "rowcta_test"):
        if MARKERS[arm][key] != expected_rowcta:
            raise SystemExit(f"{arm} rowCTA structural contract failed: {MARKERS[arm]}")
    for key in ("raster_param", "raster_dispatch", "raster_test"):
        if MARKERS[arm][key] != expected_raster:
            raise SystemExit(f"{arm} raster structural contract failed: {MARKERS[arm]}")
    if MARKERS[arm]["rejected_wave567"] or MARKERS[arm]["cp_async"]:
        raise SystemExit(f"{arm} contains a rejected or sm_80-only marker: {MARKERS[arm]}")

print(f"Notebook {NOTEBOOK_BUILD}")
print(f"Clean stack BASE + W3 {WAVE3_PATCH_SHA256} + W4 {WAVE4_PATCH_SHA256}")
print(f"W8 {WAVE8_PATCH_SHA256} | W9 {WAVE9_PATCH_SHA256} | combined {WAVE10_PATCH_SHA256}")
print(f"Run root {ROOT}")
print(json.dumps(MARKERS, indent=2))


## 2 - Fetch the pinned production model


In [ ]:
print(f"MODEL FETCH START [{NOTEBOOK_BUILD}]")
sys.stdout.flush()

import urllib.error
import urllib.request

MODEL_CACHE = WORK / "models"
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
model_dest = MODEL_CACHE / HF_FILENAME
model_part = model_dest.with_name(model_dest.name + ".part")
model_url = f"https://huggingface.co/{HF_REPO}/resolve/{HF_REVISION}/{HF_FILENAME}?download=true"

def file_sha256(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(chunk_bytes)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def park_invalid(path, reason):
    parked = path.with_name(path.name + f".invalid-{reason}-{RUN_ID}")
    path.replace(parked)
    print(f"Parked invalid cache file: {parked}")

def validate_model(path):
    if not path.is_file():
        return False, "missing"
    size = path.stat().st_size
    if size != HF_EXPECTED_BYTES:
        return False, f"size-{size}"
    with path.open("rb") as handle:
        if handle.read(4) != b"GGUF":
            return False, "magic"
    digest = file_sha256(path)
    if digest != HF_EXPECTED_SHA256:
        return False, f"sha256-{digest[:12]}"
    return True, digest

valid, detail = validate_model(model_dest)
if valid:
    print(f"Reusing verified model: {model_dest}")
else:
    if model_dest.exists():
        park_invalid(model_dest, detail)
    if model_part.exists() and model_part.stat().st_size > HF_EXPECTED_BYTES:
        park_invalid(model_part, f"oversize-{model_part.stat().st_size}")
    if model_part.exists() and model_part.stat().st_size == HF_EXPECTED_BYTES:
        partial_valid, partial_detail = validate_model(model_part)
        if partial_valid:
            model_part.replace(model_dest)
        else:
            park_invalid(model_part, partial_detail)
    if not model_dest.exists():
        start = model_part.stat().st_size if model_part.exists() else 0
        headers = {"User-Agent": "GwenLand-Wave6-Kaggle/1.0"}
        if start:
            headers["Range"] = f"bytes={start}-"
            print(f"Resuming model download at {start / 2**20:.1f} MiB")
        else:
            print(f"Downloading {HF_REPO}@{HF_REVISION[:12]}/{HF_FILENAME}")
        request = urllib.request.Request(model_url, headers=headers)
        try:
            response = urllib.request.urlopen(request, timeout=120)
        except urllib.error.HTTPError as exc:
            raise SystemExit(f"Model download HTTP {exc.code}: {exc.reason}. Confirm Kaggle Internet is On.") from exc
        except urllib.error.URLError as exc:
            raise SystemExit(f"Model download connection failed: {exc.reason}. Confirm Kaggle Internet is On.") from exc
        status = getattr(response, "status", response.getcode())
        if start and status != 206:
            print(f"Server ignored Range (HTTP {status}); restarting the partial download.")
            start = 0
        mode = "ab" if start and status == 206 else "wb"
        downloaded = start
        last_print = time.monotonic()
        with response, model_part.open(mode) as output:
            while True:
                block = response.read(8 * 1024 * 1024)
                if not block:
                    break
                output.write(block)
                downloaded += len(block)
                now = time.monotonic()
                if now - last_print >= 5:
                    pct = 100.0 * downloaded / HF_EXPECTED_BYTES
                    print(f"  {downloaded / 2**20:.1f} / {HF_EXPECTED_BYTES / 2**20:.1f} MiB ({pct:.1f}%)")
                    last_print = now
        part_valid, part_detail = validate_model(model_part)
        if not part_valid:
            park_invalid(model_part, part_detail)
            raise SystemExit(f"Downloaded GGUF failed integrity validation: {part_detail}")
        model_part.replace(model_dest)
valid, digest = validate_model(model_dest)
if not valid:
    raise SystemExit(f"Final GGUF validation failed: {digest}")
MODEL_PATH = str(model_dest)
MODEL_FETCH = {
    "repo": HF_REPO,
    "revision": HF_REVISION,
    "filename": HF_FILENAME,
    "url": model_url,
    "bytes": model_dest.stat().st_size,
    "sha256": digest,
    "path": MODEL_PATH,
}
(RESULTS / "model-fetch.json").write_text(json.dumps(MODEL_FETCH, indent=2), encoding="utf-8")
print(json.dumps(MODEL_FETCH, indent=2))


## 3 - PTX assembly, zero-spill, and occupancy gate


In [ ]:
ptxas = shutil.which("ptxas")
if ptxas is None:
    candidates = sorted(Path("/usr/local").glob("cuda*/bin/ptxas"), reverse=True)
    ptxas = str(candidates[0]) if candidates else None
if ptxas is None:
    raise SystemExit("ptxas is required for Wave 10 but was not found.")

tool_versions = {}
for name, cmd in {"nvidia_smi": ["nvidia-smi"], "rustc": ["rustc", "--version", "--verbose"], "cargo": ["cargo", "--version"], "ptxas": [ptxas, "--version"]}.items():
    p = run(cmd, check=False, timeout=120)
    tool_versions[name] = (p.stdout + p.stderr).strip()

def parse_resources(text, fn):
    header = re.search(rf"(?m)^ptxas info\s*: Function properties for {re.escape(fn)}\s*$", text)
    if not header:
        raise ValueError(f"missing Function properties for {fn}")
    tail = text[header.end():]
    nxt = re.search(r"(?m)^ptxas info\s*: Compiling entry function\b", tail)
    block = tail[:nxt.start()] if nxt else tail
    registers = re.search(r"Used\s+(\d+)\s+registers", block)
    spills = [int(x) for x in re.findall(r"(\d+) bytes spill (?:stores|loads)", block)]
    smem = re.search(r"(\d+)\s+bytes smem", block)
    if not registers or len(spills) != 2:
        raise ValueError(f"missing counters for {fn}")
    return {"registers": int(registers.group(1)), "smem_bytes": int(smem.group(1)) if smem else 0, "spill_bytes": spills}

def assemble(arm, module, functions):
    src = ARM_DIRS[arm]
    cubin = RESULTS / f"{arm}-{Path(module).stem}.cubin"
    p = run([ptxas, "-arch=sm_75", "-v", src / "glcuda/src/kernels" / module, "-o", cubin], cwd=src, timeout=600, check=False)
    save_log(f"ptxas-{arm}-{Path(module).stem}.log", p)
    text = p.stdout + "\n" + p.stderr
    if p.returncode:
        raise SystemExit(f"ptxas failed for {arm}/{module}")
    resources = {fn: parse_resources(text, fn) for fn in functions}
    if any(any(res["spill_bytes"]) for res in resources.values()):
        raise SystemExit(f"spill gate failed for {arm}/{module}: {resources}")
    return resources

PTXAS = {}
for arm in ARM_DIRS:
    main_functions = ["gl_quantize_q8", "gl_attn_decode_rows_f32", "gl_attn_rows_probe"]
    if arm in ("wave8", "wave10"):
        main_functions.append("gl_quantize_q8_rowcta")
    PTXAS[arm] = {
        "main": assemble(arm, "glcuda.ptx", main_functions),
        "sm75": assemble(arm, "glcuda_sm75.ptx", ["gl_gemm_mma_q8", "gl_gemm_mma_q8_r256"]),
    }

for arm in ARM_DIRS:
    mma = PTXAS[arm]["sm75"]["gl_gemm_mma_q8"]
    resident_ctas = min(4, 65536 // (256 * max(mma["registers"], 1)), 65536 // max(mma["smem_bytes"], 1))
    if resident_ctas * 8 < 24:
        raise SystemExit(f"{arm} MMA occupancy tier regressed: {mma}")
    if arm in ("wave8", "wave10"):
        quant = PTXAS[arm]["main"]["gl_quantize_q8_rowcta"]
        q_ctas = min(4, 65536 // (256 * max(quant["registers"], 1)), 65536 // max(quant["smem_bytes"], 1))
        if q_ctas * 8 < 24:
            raise SystemExit(f"{arm} rowCTA occupancy tier regressed: {quant}")
PTXAS_OK = True
print(json.dumps(PTXAS, indent=2))
print("PTXAS/spill/occupancy gate PASS")


## 4 - Hardware, bit-parity, and model correctness gates


In [ ]:
def cargo_env(target):
    return {"CARGO_TARGET_DIR": str(target), "RUST_BACKTRACE": "1"}

def cargo_run(label, arm, args, log_name, timeout=7200):
    p = run(["cargo", *args], cwd=ARM_DIRS[arm], env=cargo_env(ARM_TARGETS[arm]), timeout=timeout, check=False)
    save_log(log_name, p)
    hay = p.stdout + "\n" + p.stderr
    if p.returncode:
        raise SystemExit(f"{label} failed; see {log_name}")
    return hay

for arm in ARM_DIRS:
    cargo_run(f"{arm}/check", arm, ["check", "--locked", "-p", "glcuda"], f"check-{arm}.log")
    cargo_run(f"{arm}/lib", arm, ["test", "--locked", "-p", "glcuda", "--lib"], f"test-{arm}-lib.log")
    hay = cargo_run(f"{arm}/parity", arm, ["test", "--locked", "-p", "glcuda", "--test", "parity", "--", "--test-threads=1", "--nocapture"], f"test-{arm}-parity.log")
    if "SKIP: no CUDA driver/device" in hay:
        raise SystemExit(f"{arm} parity skipped the real T4")

exact_tests = {
    "wave8": ["quantize_q8_rowcta_is_byte_identical_to_the_k32_kernel"],
    "wave9": ["gemm_mma_q8_l2_raster_is_bit_identical"],
    "wave10": ["quantize_q8_rowcta_is_byte_identical_to_the_k32_kernel", "gemm_mma_q8_l2_raster_is_bit_identical"],
}
for arm, tests in exact_tests.items():
    for test in tests:
        hay = cargo_run(f"{arm}/{test}", arm, ["test", "--locked", "-p", "glcuda", "--test", "parity", test, "--", "--exact", "--nocapture", "--test-threads=1"], f"test-{arm}-{test}.log")
        if f"{test} ... ok" not in hay:
            raise SystemExit(f"{arm} exact test did not execute: {test}")

for arm in ARM_DIRS:
    cargo_run(f"{arm}/bench", arm, ["build", "--locked", "--release", "-p", "glcuda", "--example", "bench"], f"build-{arm}-bench.log")
    cargo_run(f"{arm}/glbench", arm, ["build", "--locked", "--release", "-p", "glbench"], f"build-{arm}-glbench.log")

BINS = {arm: {"bench": ARM_TARGETS[arm] / "release/examples/bench", "glbench": ARM_TARGETS[arm] / "release/glbench"} for arm in ARM_DIRS}
for arm, bins in BINS.items():
    for kind, path in bins.items():
        if not path.exists():
            raise SystemExit(f"Missing {arm}/{kind}: {path}")
CORRECTNESS_OK = True
print("Correctness gate passed for all four arms on the real T4.")


## 5 - Diagnostic combined microbenchmark


In [ ]:
if not globals().get("CORRECTNESS_OK"):
    raise SystemExit("Correctness gate did not pass.")
p = run([BINS["wave10"]["bench"]], cwd=ARM_DIRS["wave10"], env={"CUDA_VISIBLE_DEVICES": "0"}, timeout=7200, check=False)
save_log("bench-wave10.log", p)
if p.returncode:
    raise SystemExit("Wave 10 diagnostic bench failed")
hay = p.stdout + "\n" + p.stderr
row = re.search(r"\[q8-rowcta\] 244x896: warp-grid ([0-9.]+) us \| row-CTA ([0-9.]+) us \(([+-][0-9.]+)%\)", hay)
raster = re.search(r"\[gemm-l2-raster[^\]]*\].*", hay)
if not row or not raster:
    raise SystemExit("Combined bench did not emit both factorial diagnostic markers")
DIAGNOSTIC = {"rowcta": {"warp_grid_us": float(row.group(1)), "rowcta_us": float(row.group(2)), "delta_pct": float(row.group(3))}, "raster_line": raster.group(0)}
print(json.dumps(DIAGNOSTIC, indent=2))


## 6 - Latin-square production factorial

Four repeats rotate the four arms so each occupies positions 0, 1, 2, and 3 exactly once. GPU state is sampled immediately before and after every session.


In [ ]:
if not globals().get("CORRECTNESS_OK"):
    raise SystemExit("Correctness gate did not pass.")
model = Path(MODEL_PATH)
if not model.is_file() or model.stat().st_size < 10_000_000:
    raise SystemExit(f"Model path is not a plausible GGUF: {model}")

prompt_unit = ("Measure this deterministic systems prompt carefully. Explain how token-parallel integer matrix multiplication uses shared memory, Tensor Cores, and fixed launch geometry. ")
FIXED_PROMPT = prompt_unit * 8
ARMS = [
    ("wave4_attn_dsmem", "wave4", {"GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1"}),
    ("wave8_rowcta", "wave8", {"GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1", "GLCUDA_Q8_ROWCTA": "1"}),
    ("wave9_l2_raster", "wave9", {"GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1", "GLCUDA_L2_RASTER": "1"}),
    ("wave10_combined", "wave10", {"GLCUDA_FORCE_Q8": "1", "GLCUDA_GRID2D": "1", "GLCUDA_Q8_ROWCTA": "1", "GLCUDA_L2_RASTER": "1"}),
]
ARM_MAP = {label: (build, env) for label, build, env in ARMS}
LATIN_ORDER = [[ARMS[(start + offset) % 4] for offset in range(4)] for start in range(4)]

def percentile(values, q):
    values = sorted(values)
    index = (len(values) - 1) * q
    lo, hi = math.floor(index), math.ceil(index)
    return values[lo] if lo == hi else values[lo] * (hi - index) + values[hi] * (index - lo)

def session_stats(path, expected_iters=MEASURE_ITERS):
    data = json.loads(path.read_text(encoding="utf-8"))
    if "glcuda" not in json.dumps(data.get("engine", {}), sort_keys=True).lower():
        raise RuntimeError(f"Session did not record glcuda: {data.get('engine')}")
    validation = data.get("validation") or {}
    findings = validation.get("findings", [])
    parity = [f for f in findings if f.get("check") == "parity"]
    errors = [f for f in findings if f.get("severity") == "error" and f.get("check") != "parity"]
    if errors or not parity:
        raise RuntimeError(f"Invalid validation evidence: {validation}")
    match = re.search(r"(\d+)/(\d+) tokens match oracle", parity[-1].get("message", ""))
    if not match:
        raise RuntimeError(f"Could not parse oracle evidence: {parity[-1]}")
    prefix, compared = map(int, match.groups())
    if compared <= 0 or prefix != compared:
        raise RuntimeError(f"Exact glproc parity failed: {parity[-1]}")
    iterations = data.get("measurements", {}).get("iterations", [])
    prefill_ms = [float(x.get("prefill_ms", 0)) for x in iterations]
    decode_ms = [float(x.get("decode_ms", 0)) for x in iterations]
    counts = [int(x.get("prompt_tokens", 0)) for x in iterations]
    if len(iterations) != expected_iters or any(x <= 0 for x in prefill_ms + decode_ms) or len(set(counts)) != 1:
        raise RuntimeError(f"Malformed session iterations: {iterations}")
    prefill_tps = [counts[0] * 1000.0 / x for x in prefill_ms]
    decode_tps = [1000.0 / x for x in decode_ms]
    return {"prompt_tokens": counts[0], "prefill_p50": percentile(prefill_tps, .5), "prefill_mean": statistics.mean(prefill_tps), "prefill_latency_p95_ms": percentile(prefill_ms, .95), "decode_p50": percentile(decode_tps, .5), "oracle_prefix": prefix, "oracle_compared": compared, "oracle_exact": True}

def gpu_snapshot(repeat, position, arm, phase):
    query = "timestamp,index,name,pstate,temperature.gpu,power.draw,clocks.current.sm,clocks.current.memory,utilization.gpu,memory.used"
    p = run(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits", "-i", "0"], timeout=60, check=False)
    return {"repeat": repeat, "position": position, "arm": arm, "phase": phase, "returncode": p.returncode, "csv": p.stdout.strip(), "query": query}

PROD_RECORDS = []
HARDWARE_SNAPSHOTS = []
for repeat, order in enumerate(LATIN_ORDER):
    for position, (label, build_arm, extra_env) in enumerate(order):
        archive = RESULTS / f"glbench-{repeat}-{position}-{label}.json"
        HARDWARE_SNAPSHOTS.append(gpu_snapshot(repeat, position, label, "before"))
        cmd = [BINS[build_arm]["glbench"], "run", "--engine", "glcuda", "--model", model, "--prompt", FIXED_PROMPT, "--tokens", "1", "--cold-iters", str(COLD_ITERS), "--warmup", str(WARMUP_ITERS), "--iters", str(MEASURE_ITERS), "--temperature", "0", "--seed", "42", "--kind", "prefill", "--verify-against", "glproc", "--out", archive]
        p = run(cmd, cwd=ARM_DIRS[build_arm], env={"CUDA_VISIBLE_DEVICES": "0", **extra_env}, timeout=14400, check=False)
        HARDWARE_SNAPSHOTS.append(gpu_snapshot(repeat, position, label, "after"))
        save_log(f"glbench-{repeat}-{position}-{label}.log", p)
        hay = p.stdout + "\n" + p.stderr
        if p.returncode:
            raise SystemExit(f"glbench failed: {label} repeat {repeat}")
        required = ["2-D token-grid prefill GEMM enabled", "dynamic-shared prefill attention enabled", "GLCUDA_FORCE_Q8:"]
        if build_arm in ("wave8", "wave10"):
            required.append("Q8_0 row-CTA prefill quantizer enabled")
        if build_arm in ("wave9", "wave10"):
            required.append("L2-grouped GEMM raster enabled")
        forbidden = ["GLCUDA_W8PC:", "W8PC MMA enabled", "r128 prefill GEMM enabled", "r256 prefill GEMM enabled"]
        if build_arm not in ("wave8", "wave10"):
            forbidden.append("Q8_0 row-CTA prefill quantizer enabled")
        if build_arm not in ("wave9", "wave10"):
            forbidden.append("L2-grouped GEMM raster enabled")
        if any(x not in hay for x in required) or any(x in hay for x in forbidden):
            raise SystemExit(f"Dispatch contract failed for {label}; required={required}, forbidden={forbidden}")
        stats = session_stats(archive)
        PROD_RECORDS.append({"repeat": repeat, "position": position, "arm": label, "archive": str(archive), **stats})
        print(f"{label:19s} r{repeat} p{position}: {stats['prefill_p50']:.1f} tok/s | mean {stats['prefill_mean']:.1f} | P95 {stats['prefill_latency_p95_ms']:.2f} ms | oracle {stats['oracle_prefix']}/{stats['oracle_compared']}")

PROD_SUMMARY = []
for label, _, _ in ARMS:
    rows = [x for x in PROD_RECORDS if x["arm"] == label]
    PROD_SUMMARY.append({"arm": label, "session_p50_median": statistics.median(x["prefill_p50"] for x in rows), "session_mean_median": statistics.median(x["prefill_mean"] for x in rows), "session_p95_latency_median_ms": statistics.median(x["prefill_latency_p95_ms"] for x in rows), "decode_p50_median": statistics.median(x["decode_p50"] for x in rows), "sessions": len(rows), "positions": [x["position"] for x in rows]})

def paired_decision(candidate):
    paired = []
    for repeat in range(PRODUCTION_REPEATS):
        base = next(x for x in PROD_RECORDS if x["repeat"] == repeat and x["arm"] == "wave4_attn_dsmem")
        cand = next(x for x in PROD_RECORDS if x["repeat"] == repeat and x["arm"] == candidate)
        paired.append({"repeat": repeat, "prefill_p50_delta": cand["prefill_p50"] / base["prefill_p50"] - 1, "prefill_mean_delta": cand["prefill_mean"] / base["prefill_mean"] - 1, "p95_latency_delta": cand["prefill_latency_p95_ms"] / base["prefill_latency_p95_ms"] - 1, "decode_p50_delta": cand["decode_p50"] / base["decode_p50"] - 1})
    summary = {x["arm"]: x for x in PROD_SUMMARY}
    median_delta = summary[candidate]["session_p50_median"] / summary["wave4_attn_dsmem"]["session_p50_median"] - 1
    return {"candidate": candidate, "median_delta": median_delta, "paired": paired, "prefill_p50_pass": all(x["prefill_p50_delta"] >= .05 for x in paired), "prefill_mean_pass": all(x["prefill_mean_delta"] >= .05 for x in paired), "tail_pass": all(x["p95_latency_delta"] <= .05 for x in paired), "decode_pass": all(x["decode_p50_delta"] >= -.05 for x in paired), "oracle_pass": all(x["oracle_exact"] for x in PROD_RECORDS if x["arm"] == candidate)}

DECISIONS = {label: paired_decision(label) for label, _, _ in ARMS[1:]}
for d in DECISIONS.values():
    d["retain"] = d["median_delta"] >= .05 and d["prefill_p50_pass"] and d["prefill_mean_pass"] and d["tail_pass"] and d["decode_pass"] and d["oracle_pass"]
summary_map = {x["arm"]: x for x in PROD_SUMMARY}
BEST_ARM = max(PROD_SUMMARY, key=lambda x: x["session_p50_median"])["arm"]
base_tps = summary_map["wave4_attn_dsmem"]["session_p50_median"]
r8 = summary_map["wave8_rowcta"]["session_p50_median"] / base_tps
r9 = summary_map["wave9_l2_raster"]["session_p50_median"] / base_tps
r10 = summary_map["wave10_combined"]["session_p50_median"] / base_tps
FACTORIAL = {"rowcta_main_effect": r8 - 1, "raster_main_effect": r9 - 1, "combined_effect": r10 - 1, "multiplicative_interaction": r10 / (r8 * r9) - 1, "best_arm": BEST_ARM}

best_build, best_env = ARM_MAP[BEST_ARM]
telemetry_archive = RESULTS / f"telemetry-{BEST_ARM}.json"
p = run([BINS[best_build]["glbench"], "run", "--engine", "glcuda", "--model", model, "--prompt", FIXED_PROMPT, "--tokens", "1", "--cold-iters", "0", "--warmup", "3", "--iters", "1", "--temperature", "0", "--seed", "42", "--kind", "prefill", "--verify-against", "glproc", "--out", telemetry_archive], cwd=ARM_DIRS[best_build], env={"CUDA_VISIBLE_DEVICES": "0", **best_env, "GLCUDA_TELEMETRY": "1"}, timeout=14400, check=False)
save_log(f"telemetry-{BEST_ARM}.log", p)
if p.returncode:
    raise SystemExit("Best-arm telemetry failed")
telemetry_data = json.loads(telemetry_archive.read_text(encoding="utf-8"))
STAGES = (((telemetry_data.get("telemetry") or {}).get("prefill") or {}).get("stages") or [])
if not STAGES:
    raise SystemExit("Best-arm telemetry emitted no stages")
stage_total = sum(float(x.get("total_ms") or 0) for x in STAGES)
attention_ms = sum(float(x.get("total_ms") or 0) for x in STAGES if x.get("name") == "attention")
gemm_ms = sum(float(x.get("total_ms") or 0) for x in STAGES if x.get("name") in {"qkv", "attn_out", "ffn_gate_up", "ffn_down"})
best_tps = summary_map[BEST_ARM]["session_p50_median"]
prompt_tokens = next(x["prompt_tokens"] for x in PROD_RECORDS if x["arm"] == BEST_ARM)
TARGET_ANALYSIS = {"best_arm": BEST_ARM, "measured_tps": best_tps, "target_tps": TARGET_PREFILL_TPS, "required_speedup": TARGET_PREFILL_TPS / best_tps, "measured_prefill_ms": prompt_tokens * 1000 / best_tps, "target_prefill_ms": prompt_tokens * 1000 / TARGET_PREFILL_TPS, "attention_share": attention_ms / stage_total, "gemm_share": gemm_ms / stage_total, "infinite_attention_ceiling_tps": best_tps / (1 - attention_ms / stage_total), "infinite_gemm_ceiling_tps": best_tps / (1 - gemm_ms / stage_total)}

for repeat in range(PRODUCTION_REPEATS):
    base = next(Path(x["archive"]) for x in PROD_RECORDS if x["repeat"] == repeat and x["arm"] == "wave4_attn_dsmem")
    for label, build, _ in ARMS[1:]:
        cand = next(Path(x["archive"]) for x in PROD_RECORDS if x["repeat"] == repeat and x["arm"] == label)
        p = run([BINS[build]["glbench"], "compare", base, cand], cwd=ARM_DIRS[build], timeout=600, check=False)
        save_log(f"compare-{repeat}-wave4-vs-{label}.log", p)
        if p.returncode:
            raise SystemExit(f"glbench compare failed: {label} repeat {repeat}")
PROD_OK = True
print(json.dumps(PROD_SUMMARY, indent=2))
print(json.dumps(FACTORIAL, indent=2))
print(json.dumps(DECISIONS, indent=2))


## 7 - Optional Nsight Compute evidence


In [ ]:
NCU = {"available": False, "attempted": False, "permission_denied": False}
ncu = shutil.which("ncu")
if RUN_NCU and ncu:
    NCU["available"] = True
    NCU["attempted"] = True
    p = run([ncu, "--target-processes", "all", "--set", "basic", BINS["wave10"]["bench"]], cwd=ARM_DIRS["wave10"], env={"CUDA_VISIBLE_DEVICES": "0", "GLCUDA_Q8_ROWCTA": "1", "GLCUDA_L2_RASTER": "1"}, timeout=1800, check=False)
    save_log("ncu-wave10.log", p)
    text = p.stdout + "\n" + p.stderr
    NCU["returncode"] = p.returncode
    NCU["permission_denied"] = "ERR_NVGPUCTRPERM" in text
    NCU["tail"] = text[-3000:]
elif RUN_NCU:
    NCU["reason"] = "ncu not installed"
print(json.dumps(NCU, indent=2))


## 8 - Package the Wave 10 evidence


In [ ]:
if not globals().get("PROD_OK"):
    raise SystemExit("Production gate did not complete.")
manifest = {
    "schema": "gwenland.glcuda.t4-ceiling.wave10.factorial.v1",
    "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "notebook": NOTEBOOK_BUILD, "gpu": gpu_rows, "repo_url": REPO_URL, "base_rev": BASE_REV,
    "patches": {"wave3": WAVE3_PATCH_SHA256, "wave4": WAVE4_PATCH_SHA256, "wave8": WAVE8_PATCH_SHA256, "wave9": WAVE9_PATCH_SHA256, "wave10_combined": WAVE10_PATCH_SHA256},
    "model": {"repo": HF_REPO, "revision": HF_REVISION, "filename": HF_FILENAME, "bytes": HF_EXPECTED_BYTES, "sha256": HF_EXPECTED_SHA256},
    "protocol": {"latin_order": [[x[0] for x in order] for order in LATIN_ORDER], "repeats": PRODUCTION_REPEATS, "cold": COLD_ITERS, "warmup": WARMUP_ITERS, "measure": MEASURE_ITERS},
    "markers": MARKERS, "ptxas": PTXAS, "diagnostic": DIAGNOSTIC, "production_records": PROD_RECORDS, "production_summary": PROD_SUMMARY,
    "hardware_snapshots": HARDWARE_SNAPSHOTS, "factorial": FACTORIAL, "decisions": DECISIONS, "target_analysis": TARGET_ANALYSIS, "ncu": NCU, "tool_versions": tool_versions,
}
(RESULTS / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

lines = [
    "# glcuda T4 Ceiling - Wave 10", "", f"- notebook: {NOTEBOOK_BUILD}", f"- GPU: {gpu_rows[0]}", f"- baseline revision: {BASE_REV}",
    f"- Wave 3 patch: {WAVE3_PATCH_SHA256}", f"- retained Wave 4 patch: {WAVE4_PATCH_SHA256}", f"- Wave 8 patch: {WAVE8_PATCH_SHA256}", f"- Wave 9 patch: {WAVE9_PATCH_SHA256}", f"- combined patch: {WAVE10_PATCH_SHA256}",
    "- rejected Wave 5/6/7 patches applied: NO", "- ptxas: PASS", "- hardware correctness: PASS", "- exact glproc parity: PASS for every production session", f"- production protocol: {PRODUCTION_REPEATS}x4 Latin-square", f"- target: {TARGET_PREFILL_TPS:.0f} prefill tok/s", "",
    "## Production prefill", "", "| arm | median session P50 tok/s | median mean | median P95 latency ms | decode P50 | sessions |", "|---|---:|---:|---:|---:|---:|",
]
for row in PROD_SUMMARY:
    lines.append(f"| {row['arm']} | {row['session_p50_median']:.1f} | {row['session_mean_median']:.1f} | {row['session_p95_latency_median_ms']:.2f} | {row['decode_p50_median']:.1f} | {row['sessions']} |")
lines += ["", "## Factorial result", "", f"- rowCTA main effect: {FACTORIAL['rowcta_main_effect']:+.2%}", f"- L2 raster main effect: {FACTORIAL['raster_main_effect']:+.2%}", f"- combined effect: {FACTORIAL['combined_effect']:+.2%}", f"- multiplicative interaction: {FACTORIAL['multiplicative_interaction']:+.2%}", f"- fastest valid arm: {BEST_ARM}", ""]
for label, decision in DECISIONS.items():
    lines.append(f"- {label}: {decision['median_delta']:+.2%}, verdict {'RETAIN' if decision['retain'] else 'HOLD/REJECT'}")
lines += ["", "## 15k feasibility budget", "", f"- measured/target: {TARGET_ANALYSIS['measured_tps']:.1f} / {TARGET_ANALYSIS['target_tps']:.0f} tok/s", f"- required speedup: {TARGET_ANALYSIS['required_speedup']:.2f}x", f"- measured/target prompt time: {TARGET_ANALYSIS['measured_prefill_ms']:.2f} / {TARGET_ANALYSIS['target_prefill_ms']:.2f} ms", f"- attention/GEMM stage share: {TARGET_ANALYSIS['attention_share']:.1%} / {TARGET_ANALYSIS['gemm_share']:.1%}", f"- infinite-attention-only ceiling: {TARGET_ANALYSIS['infinite_attention_ceiling_tps']:.1f} tok/s", f"- infinite-GEMM-only ceiling: {TARGET_ANALYSIS['infinite_gemm_ceiling_tps']:.1f} tok/s", "", "## Interpretation rule", "", "Retain an arm only when every paired block improves prefill P50 and mean by at least 5%, P95 latency and decode remain within 5%, exact oracle parity stays green, and PTXAS reports zero spills without falling below the 24-warp resource tier. The Latin square makes every arm occupy every run position once. Diagnostic microbenchmarks and the singleton stage-telemetry run are not retention gates."]
(RESULTS / "report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
archive_base = WORK / "glcuda_t4_ceiling_wave10_fetch_results"
archive = Path(shutil.make_archive(str(archive_base), "zip", root_dir=RESULTS))
print((RESULTS / "report.md").read_text(encoding="utf-8"))
print(f"Results directory: {RESULTS}")
print(f"Download archive: {archive}")
print(f"Archive size: {archive.stat().st_size / 1_000_000:.2f} MB")
